# Task 2 — Corruption Classification and Hard-Routed Specialist Autoencoders

$$\hat{x} = \begin{cases} \tilde{x} & r=\text{clean (identity bypass)}\\ A_{\text{salt}}(\tilde{x}) & r=\text{salt}\\ A_{\text{blur}}(\tilde{x}) & r=\text{blur}\\ A_{\text{occ}}(\tilde{x}) & r=\text{occlusion}\end{cases}\qquad r=\arg\max_k p_k,\; p=C(\tilde{x})$$

## ⚠️ Run in this order
1. **`MODE = "smoke"`** (the default) → *Run All* in the interactive editor. It takes about 10–15 minutes on a T4.
   The pre-flight gate runs first, then every stage on tiny subsets. The notebook ends with `SMOKE TEST PASSED`. Check the W&B group `task2-smoke`.
2. **`MODE = "full"`** → **Save Version → Save & Run All**. It takes about 3–3.5 hours, and the runs appear live in the W&B group `task2`.

**Kaggle settings (verified by the pre-flight gate before anything trains):**
GPU **T4** · Internet **On** · *Add-ons → Secrets →* `WANDB_API_KEY` **ticked for this notebook** ·
Inputs: the Oxford-IIIT Pet dataset + the **notebook 00 output** + (recommended) the **Task 1 v2 output**, used for the universal-vs-routed comparison.

| Section | Content |
|---|---|
| 0 | Setup, modules, config, W&B |
| 1 | Data + **pre-flight gate** |
| 2 | Unit tests (classifier, balanced batches, specialists, routing, ONNX) |
| 3 | **Classifier**: Optuna (lr, batch size, channel configuration, dropout, weight decay) → final training |
| 4 | Classifier evaluation: accuracy, macro P/R/F1, per-class metrics, normalised confusion matrices (val + test), accuracy by severity |
| 5 | **Specialists**: one shared Optuna search (lr, bottleneck, channels, batch size, L1/SSIM weight) → 3 independently trained experts |
| 6 | **Hard-routed restoration on the test set**: oracle vs predicted routing, cross-expert matrix, misrouting cost |
| 7 | Figures: representative routed examples and classifier-induced failures |
| 8 | ONNX export of the classifier and all 3 specialists (parity, latency, sidecar JSON) |
| 9 | Summary |

## 0. Setup

In [ ]:
import importlib.util, subprocess, sys
missing = [p for p in ("onnx", "onnxruntime", "optuna", "sklearn") if importlib.util.find_spec(p) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *[("scikit-learn" if m == "sklearn" else m) for m in missing]], check=False)
print("installed:", missing or "nothing needed")

In [ ]:
%%writefile pet_data.py
"""
pet_data.py — shared data pipeline for Tasks 1–3 (Oxford-IIIT Pet restoration).

Single source of truth for:
  * locating the Oxford-IIIT Pet dataset (Kaggle / torchvision layouts),
  * loading + standardising clean images (RGB, 128x128) with an on-disk cache
    of the *clean* resized images only (no corrupted copies are ever written),
  * the deterministic 80/20 train/val split of the official `trainval` list (seed 42),
  * the four input conditions and their corruption functions (pure NumPy, so the
    FastAPI backend can import them without PyTorch),
  * validation / test corruption manifests (deterministic, fully parameterised),
  * PyTorch datasets, a class-balanced batch sampler and DataLoader builders.

Label convention (used everywhere, incl. the Task 3 gate weight order w0..w3):
    0 = clean, 1 = salt_pepper, 2 = gaussian_blur, 3 = occlusion
"""
from __future__ import annotations

import hashlib
import json
import os
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Iterable, Sequence

import numpy as np
from PIL import Image

# --------------------------------------------------------------------------------------
# Constants (the assignment's corruption specification)
# --------------------------------------------------------------------------------------
IMG_SIZE = 128
SPLIT_SEED = 42
VAL_FRACTION = 0.20

CLEAN, SALT_PEPPER, BLUR, OCCLUSION = 0, 1, 2, 3
CLASS_NAMES = ("clean", "salt_pepper", "gaussian_blur", "occlusion")
NUM_CLASSES = 4

SP_PROB_RANGE = (0.02, 0.15)          # uniform
BLUR_KERNELS = (3, 5, 7)              # uniform choice
BLUR_SIGMA_RANGE = (0.5, 2.5)         # uniform
OCC_BOX_RANGE = (1, 3)                # inclusive, uniform
OCC_COVERAGE_RANGE = (0.10, 0.35)     # union of boxes / image area, uniform
OCC_ASPECT_RANGE = (0.5, 2.0)         # box width/height, log-uniform (design choice)
OCC_MIN_SIDE = 4                      # px (design choice)
TEST_OCC_TOLERANCE = 0.01             # |actual - target| coverage for fixed test boxes

SEVERITY_LEVELS = ("low", "medium", "high")
TEST_SEVERITIES = {
    SALT_PEPPER: {"low": {"p": 0.03}, "medium": {"p": 0.08}, "high": {"p": 0.15}},
    BLUR: {"low": {"kernel_size": 3, "sigma": 0.7},
           "medium": {"kernel_size": 5, "sigma": 1.5},
           "high": {"kernel_size": 7, "sigma": 2.5}},
    OCCLUSION: {"low": {"n_boxes": 1, "coverage": 0.10},
                "medium": {"n_boxes": 2, "coverage": 0.20},
                "high": {"n_boxes": 3, "coverage": 0.35}},
}

MANIFEST_VERSION = 1


def corruption_config() -> dict:
    """The full corruption configuration as a JSON-serialisable dict (for W&B / report)."""
    return {
        "image_size": IMG_SIZE,
        "class_names": list(CLASS_NAMES),
        "train_class_probs": [0.25] * 4,
        "salt_pepper": {"p_uniform": list(SP_PROB_RANGE), "salt_vs_pepper": 0.5,
                        "granularity": "per-pixel (all 3 channels)"},
        "gaussian_blur": {"kernel_sizes": list(BLUR_KERNELS), "sigma_uniform": list(BLUR_SIGMA_RANGE),
                          "padding": "reflect", "separable": True},
        "occlusion": {"n_boxes": list(OCC_BOX_RANGE), "coverage_uniform": list(OCC_COVERAGE_RANGE),
                      "aspect_log_uniform": list(OCC_ASPECT_RANGE), "min_side_px": OCC_MIN_SIDE,
                      "boxes_overlap": False, "fill": 0.0},
        "test_severities": {CLASS_NAMES[k]: v for k, v in TEST_SEVERITIES.items()},
        "test_occlusion_tolerance": TEST_OCC_TOLERANCE,
    }


# --------------------------------------------------------------------------------------
# Dataset discovery and loading
# --------------------------------------------------------------------------------------
def read_split_list(path: str | Path) -> list[str]:
    """Read an official list file (e.g. 'Abyssinian_100 1 1 1'). Breed/species ids are discarded."""
    names = []
    for line in Path(path).read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#"):
            names.append(line.split()[0])
    return names


def find_pet_dataset(search_root: str | Path = "/kaggle/input") -> dict:
    """Locate annotations (trainval.txt + test.txt) and the images folder under `search_root`.

    Works for the common Kaggle upload (images/images, annotations/annotations) and the
    torchvision layout (oxford-iiit-pet/images, oxford-iiit-pet/annotations).
    """
    search_root = Path(search_root)
    ann_dir = None
    for tv in sorted(search_root.rglob("trainval.txt")):
        if (tv.parent / "test.txt").exists():
            ann_dir = tv.parent
            break
    if ann_dir is None:
        raise FileNotFoundError(
            f"No folder containing both trainval.txt and test.txt under {search_root}. "
            "The official split lists are required; set IMAGES_DIR / ANNOT_DIR manually.")
    first = read_split_list(ann_dir / "trainval.txt")[0]
    hits = sorted(search_root.rglob(f"{first}.jpg"))
    if not hits:
        raise FileNotFoundError(f"Could not find image '{first}.jpg' under {search_root}.")
    return {"annotations_dir": ann_dir, "images_dir": hits[0].parent}


def load_standardised_image(path: str | Path, size: int = IMG_SIZE) -> tuple[np.ndarray, str]:
    """Open any image, force 3-channel RGB, resize to size x size (bicubic, antialiased).

    Returns (uint8 HxWx3 array, original PIL mode). The backend must use this same function.
    """
    with Image.open(path) as im:
        mode = im.mode
        im = im.convert("RGB").resize((size, size), Image.Resampling.BICUBIC)
        return np.asarray(im, dtype=np.uint8).copy(), mode


def load_clean_images(names: Sequence[str], images_dir: str | Path, cache_path: str | Path | None = None,
                      size: int = IMG_SIZE, num_threads: int = 8, verbose: bool = True) -> dict:
    """Load + standardise every image in `names`. Caches the CLEAN resized uint8 array to .npz.

    Returns dict(images=(N,H,W,3) uint8, names=[...kept...], failed=[...], modes=Counter).
    Images that cannot be decoded are reported and dropped (the split is made afterwards).
    """
    names = list(names)
    if cache_path is not None and Path(cache_path).exists():
        z = np.load(cache_path, allow_pickle=False)
        if list(z["requested"]) == names and int(z["size"]) == size:
            out = {"images": z["images"], "names": list(z["names"]), "failed": list(z["failed"]),
                   "modes": Counter(json.loads(str(z["modes"]))), "from_cache": True}
            if verbose:
                print(f"Loaded {len(out['names'])} cached images from {cache_path}")
            return out

    images_dir = Path(images_dir)

    def _load(name):
        try:
            arr, mode = load_standardised_image(images_dir / f"{name}.jpg", size)
            return name, arr, mode, None
        except Exception as e:  # corrupted / unreadable file
            return name, None, None, repr(e)

    with ThreadPoolExecutor(max_workers=num_threads) as ex:
        results = list(ex.map(_load, names))

    kept = [(n, a, m) for n, a, m, err in results if err is None]
    failed = [(n, err) for n, a, m, err in results if err is not None]
    imgs = np.stack([a for _, a, _ in kept]) if kept else np.zeros((0, size, size, 3), np.uint8)
    out = {"images": imgs, "names": [n for n, _, _ in kept], "failed": [n for n, _ in failed],
           "modes": Counter(m for _, _, m in kept), "from_cache": False}
    if verbose:
        print(f"Loaded {len(kept)}/{len(names)} images; failed: {len(failed)}")
        for n, err in failed:
            print(f"  ! {n}: {err}")
    if cache_path is not None:
        Path(cache_path).parent.mkdir(parents=True, exist_ok=True)
        np.savez(cache_path, images=imgs, names=np.array(out["names"]), requested=np.array(names),
                 failed=np.array(out["failed"], dtype=str), size=size, modes=json.dumps(dict(out["modes"])))
    return out


# --------------------------------------------------------------------------------------
# Deterministic split
# --------------------------------------------------------------------------------------
def make_split(names: Sequence[str], val_fraction: float = VAL_FRACTION, seed: int = SPLIT_SEED) -> dict:
    """80/20 random split of the official trainval list with numpy default_rng(seed)."""
    names = list(names)
    n = len(names)
    perm = np.random.default_rng(seed).permutation(n)
    n_val = int(round(val_fraction * n))
    n_train = n - n_val
    train = [names[i] for i in sorted(perm[:n_train])]
    val = [names[i] for i in sorted(perm[n_train:])]
    return {"seed": seed, "val_fraction": val_fraction, "method": "numpy.default_rng(seed).permutation",
            "n_total": n, "train": train, "val": val}


# --------------------------------------------------------------------------------------
# Corruption functions (pure NumPy; img = float32 HxWx3 in [0,1])
# --------------------------------------------------------------------------------------
def salt_and_pepper(img: np.ndarray, p: float, seed: int) -> np.ndarray:
    """Each pixel is selected with prob p; selected pixels become white or black (50/50)."""
    rng = np.random.default_rng(seed)
    h, w = img.shape[:2]
    selected = rng.random((h, w)) < p
    salt = rng.random((h, w)) < 0.5
    out = img.copy()
    out[selected & salt] = 1.0
    out[selected & ~salt] = 0.0
    return out


def gaussian_kernel1d(kernel_size: int, sigma: float) -> np.ndarray:
    x = np.arange(kernel_size, dtype=np.float64) - (kernel_size - 1) / 2.0
    k = np.exp(-0.5 * (x / sigma) ** 2)
    return (k / k.sum()).astype(np.float32)


def gaussian_blur(img: np.ndarray, kernel_size: int, sigma: float) -> np.ndarray:
    """Separable Gaussian blur with reflect padding (matches torchvision's gaussian_blur)."""
    k = gaussian_kernel1d(kernel_size, sigma)
    r = kernel_size // 2
    h, w = img.shape[:2]
    padded = np.pad(img, ((r, r), (r, r), (0, 0)), mode="reflect")
    horiz = sum(k[i] * padded[:, i:i + w] for i in range(kernel_size))        # (h+2r, w, 3)
    out = sum(k[i] * horiz[i:i + h] for i in range(kernel_size))              # (h, w, 3)
    return np.clip(out, 0.0, 1.0).astype(np.float32)


def occlude(img: np.ndarray, boxes: Iterable[Sequence[int]], fill: float = 0.0) -> np.ndarray:
    """Black out boxes given as [x0, y0, x1, y1] (x1/y1 exclusive)."""
    out = img.copy()
    for x0, y0, x1, y1 in boxes:
        out[y0:y1, x0:x1] = fill
    return out


def boxes_coverage(boxes, h: int = IMG_SIZE, w: int = IMG_SIZE) -> float:
    mask = np.zeros((h, w), bool)
    for x0, y0, x1, y1 in boxes:
        mask[y0:y1, x0:x1] = True
    return float(mask.mean())


def sample_occlusion_boxes(rng: np.random.Generator, n_boxes: int, coverage: float,
                           h: int = IMG_SIZE, w: int = IMG_SIZE,
                           accept=None, max_attempts: int = 1000) -> tuple[list[list[int]], float]:
    """Sample `n_boxes` non-overlapping boxes whose union covers ~`coverage` of the image.

    The total area is split between boxes (each gets at least half of an equal share), each
    box gets a log-uniform aspect ratio, and boxes are placed at random non-overlapping
    positions. Non-overlap makes the union area equal to the sum of box areas, so the jointly
    covered fraction is controlled precisely. `accept(actual)->bool` decides if rounding is OK.
    """
    if accept is None:
        accept = lambda a: abs(a - coverage) <= TEST_OCC_TOLERANCE  # noqa: E731
    total = coverage * h * w
    lo_ar, hi_ar = np.log(OCC_ASPECT_RANGE[0]), np.log(OCC_ASPECT_RANGE[1])
    for _ in range(max_attempts):
        shares = 0.5 / n_boxes + 0.5 * rng.dirichlet([2.0] * n_boxes)
        boxes, ok = [], True
        for area in shares * total:
            ar = float(np.exp(rng.uniform(lo_ar, hi_ar)))
            bw = int(np.clip(round(np.sqrt(area * ar)), OCC_MIN_SIDE, w))
            bh = int(np.clip(round(area / bw), OCC_MIN_SIDE, h))
            for _try in range(200):
                x0 = int(rng.integers(0, w - bw + 1))
                y0 = int(rng.integers(0, h - bh + 1))
                cand = [x0, y0, x0 + bw, y0 + bh]
                if all(cand[2] <= b[0] or b[2] <= cand[0] or cand[3] <= b[1] or b[3] <= cand[1]
                       for b in boxes):
                    boxes.append(cand)
                    break
            else:
                ok = False
                break
        if ok:
            actual = boxes_coverage(boxes, h, w)
            if accept(actual):
                return boxes, actual
    raise RuntimeError(f"Could not place {n_boxes} boxes covering {coverage:.3f}")


def _new_seed(rng: np.random.Generator) -> int:
    return int(rng.integers(0, 2**31 - 1))


def sample_params(label: int, rng: np.random.Generator) -> dict:
    """Sample corruption parameters from the TRAINING distribution. Fully reproducible via 'seed'."""
    seed = _new_seed(rng)
    r = np.random.default_rng(seed)
    label = int(label)
    if label == CLEAN:
        return {"label": CLEAN, "type": "clean", "seed": seed}
    if label == SALT_PEPPER:
        return {"label": SALT_PEPPER, "type": "salt_pepper", "seed": seed,
                "p": float(r.uniform(*SP_PROB_RANGE))}
    if label == BLUR:
        return {"label": BLUR, "type": "gaussian_blur", "seed": seed,
                "kernel_size": int(r.choice(BLUR_KERNELS)), "sigma": float(r.uniform(*BLUR_SIGMA_RANGE))}
    if label == OCCLUSION:
        n = int(r.integers(OCC_BOX_RANGE[0], OCC_BOX_RANGE[1] + 1))
        cov = float(r.uniform(*OCC_COVERAGE_RANGE))
        lo, hi = OCC_COVERAGE_RANGE
        boxes, actual = sample_occlusion_boxes(r, n, cov, accept=lambda a: lo <= a <= hi)
        return {"label": OCCLUSION, "type": "occlusion", "seed": seed, "n_boxes": n,
                "coverage_target": cov, "coverage": actual, "boxes": boxes}
    raise ValueError(f"Unknown label {label}")


def fixed_params(label: int, level: str, seed: int) -> dict:
    """Parameters for the fixed TEST severities (low/medium/high)."""
    label = int(label)
    if label == CLEAN:
        return {"label": CLEAN, "type": "clean", "seed": seed}
    spec = TEST_SEVERITIES[label][level]
    base = {"label": label, "type": CLASS_NAMES[label], "seed": seed}
    if label == SALT_PEPPER:
        return {**base, "p": spec["p"]}
    if label == BLUR:
        return {**base, "kernel_size": spec["kernel_size"], "sigma": spec["sigma"]}
    boxes, actual = sample_occlusion_boxes(np.random.default_rng(seed), spec["n_boxes"], spec["coverage"])
    return {**base, "n_boxes": spec["n_boxes"], "coverage_target": spec["coverage"],
            "coverage": actual, "boxes": boxes}


def apply_corruption(img: np.ndarray, params: dict) -> np.ndarray:
    """Deterministically apply a parameter dict (from sample_params / fixed_params / a manifest)."""
    label = int(params["label"])
    if label == CLEAN:
        return img.copy()
    if label == SALT_PEPPER:
        return salt_and_pepper(img, params["p"], params["seed"])
    if label == BLUR:
        return gaussian_blur(img, params["kernel_size"], params["sigma"])
    if label == OCCLUSION:
        return occlude(img, params["boxes"])
    raise ValueError(f"Unknown label {label}")


def describe_params(params: dict) -> str:
    """Short human-readable description (plot titles, API responses)."""
    t = int(params["label"])
    if t == CLEAN:
        return "clean"
    if t == SALT_PEPPER:
        return f"s&p p={params['p']:.3f}"
    if t == BLUR:
        return f"blur k={params['kernel_size']} σ={params['sigma']:.2f}"
    return f"occl {params['n_boxes']}box {100 * params['coverage']:.1f}%"


# --------------------------------------------------------------------------------------
# Manifests
# --------------------------------------------------------------------------------------
def build_val_manifest(val_names: Sequence[str], seed: int = SPLIT_SEED) -> dict:
    """Exactly balanced (25% per condition) validation manifest; params from the training ranges."""
    rng = np.random.default_rng(np.random.SeedSequence([seed, 1]))
    n = len(val_names)
    labels = rng.permutation(np.resize(np.arange(NUM_CLASSES), n))
    records = []
    for i, (name, label) in enumerate(zip(val_names, labels)):
        p = sample_params(int(label), rng)
        records.append({"id": f"{name}__{p['type']}", "image": name, "index": i,
                        "severity_level": "sampled" if label != CLEAN else "none", **p})
    return {"meta": {"kind": "validation", "version": MANIFEST_VERSION, "seed": seed,
                     "seed_sequence": [seed, 1], "n_images": n, "n_records": len(records),
                     "config": corruption_config()},
            "records": records}


def build_test_manifest(test_names: Sequence[str], seed: int = SPLIT_SEED) -> dict:
    """Every test image × {clean, 3 corruptions × 3 fixed severities} = 10 records per image."""
    rng = np.random.default_rng(np.random.SeedSequence([seed, 2]))
    records = []
    for i, name in enumerate(test_names):
        s = _new_seed(rng)
        records.append({"id": f"{name}__clean", "image": name, "index": i,
                        "severity_level": "none", **fixed_params(CLEAN, "none", s)})
        for label in (SALT_PEPPER, BLUR, OCCLUSION):
            for level in SEVERITY_LEVELS:
                s = _new_seed(rng)
                p = fixed_params(label, level, s)
                records.append({"id": f"{name}__{p['type']}__{level}", "image": name, "index": i,
                                "severity_level": level, **p})
    return {"meta": {"kind": "test", "version": MANIFEST_VERSION, "seed": seed,
                     "seed_sequence": [seed, 2], "n_images": len(test_names), "n_records": len(records),
                     "config": corruption_config()},
            "records": records}


def save_json(obj, path: str | Path, indent=None) -> str:
    """Write JSON deterministically and return its SHA-256 (use it to prove files are identical)."""
    text = json.dumps(obj, indent=indent, sort_keys=True, separators=None if indent else (",", ":"))
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(text)
    return hashlib.sha256(text.encode()).hexdigest()


def load_json(path: str | Path):
    return json.loads(Path(path).read_text())


def sha256_file(path: str | Path) -> str:
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


# --------------------------------------------------------------------------------------
# PyTorch datasets / samplers / loaders  (torch imported lazily so the backend stays light)
# --------------------------------------------------------------------------------------
try:
    import torch
    from torch.utils.data import DataLoader, Dataset, Sampler, get_worker_info
except ImportError:  # backend without torch: only the NumPy functions above are needed
    torch = None
    Dataset = object
    Sampler = object


def to_chw_tensor(img_hwc: np.ndarray):
    return torch.from_numpy(np.ascontiguousarray(img_hwc.transpose(2, 0, 1), dtype=np.float32))


class RuntimeCorruptionDataset(Dataset):
    """Training dataset: a NEW corruption type + severity is sampled on every __getitem__.

    Returns (corrupted_image, clean_image, corruption_label):
        (3,128,128) float32 [0,1], (3,128,128) float32 [0,1], int64 scalar.
    `classes` restricts which conditions are drawn (e.g. (1,) for the Task 2 salt-and-pepper
    specialist); they are drawn uniformly. `item` may also be an (index, label) tuple, which is
    how BalancedCorruptionBatchSampler forces exactly balanced batches.
    """

    def __init__(self, images_u8: np.ndarray, classes: Sequence[int] = (0, 1, 2, 3), seed: int | None = None):
        self.images = images_u8
        self.classes = np.asarray(classes, dtype=np.int64)
        self.seed = seed
        self._rng, self._rng_key = None, None

    def __len__(self):
        return len(self.images)

    def _get_rng(self) -> np.random.Generator:
        # One independent generator per process/worker. Worker seeds come from torch
        # (base_seed + worker_id, re-drawn every epoch), so torch.manual_seed controls everything
        # and forked workers never share a random stream.
        info = get_worker_info()
        key = (os.getpid(), info.seed if info is not None else None)
        if key != self._rng_key:
            if info is not None:
                s = info.seed % 2**32
            else:
                s = self.seed if self.seed is not None else int(torch.initial_seed() % 2**32)
            self._rng, self._rng_key = np.random.default_rng(s), key
        return self._rng

    def get_with_params(self, item):
        rng = self._get_rng()
        if isinstance(item, (tuple, list)):
            idx, label = int(item[0]), int(item[1])
        else:
            idx, label = int(item), int(rng.choice(self.classes))
        clean = self.images[idx].astype(np.float32) / 255.0
        params = sample_params(label, rng)
        corrupted = apply_corruption(clean, params)
        return corrupted, clean, params

    def __getitem__(self, item):
        corrupted, clean, params = self.get_with_params(item)
        return to_chw_tensor(corrupted), to_chw_tensor(clean), torch.tensor(params["label"], dtype=torch.long)


class ManifestCorruptionDataset(Dataset):
    """Validation/test dataset: corruption parameters are read from a fixed manifest.

    `name_to_row` maps image name -> row in `images_u8`. With return_index=True a 4th element
    (the record index) is returned so results can be grouped by type/severity afterwards.
    """

    def __init__(self, images_u8: np.ndarray, name_to_row: dict, records: list[dict],
                 classes: Sequence[int] | None = None, return_index: bool = False):
        self.images = images_u8
        self.name_to_row = name_to_row
        self.records = [r for r in records if classes is None or r["label"] in classes]
        self.return_index = return_index

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        rec = self.records[i]
        clean = self.images[self.name_to_row[rec["image"]]].astype(np.float32) / 255.0
        corrupted = apply_corruption(clean, rec)
        out = (to_chw_tensor(corrupted), to_chw_tensor(clean), torch.tensor(rec["label"], dtype=torch.long))
        return out + (torch.tensor(i),) if self.return_index else out


class BalancedCorruptionBatchSampler(Sampler):
    """Yields batches of (image_index, label) with EXACTLY equal counts per class
    (when batch_size is a multiple of the number of classes). Intended for the Task 2 classifier.
    """

    def __init__(self, n_items: int, batch_size: int, classes: Sequence[int] = (0, 1, 2, 3),
                 drop_last: bool = True, seed: int | None = None):
        self.n, self.bs, self.classes, self.drop_last = n_items, batch_size, np.asarray(classes), drop_last
        self.seed, self.epoch = seed, 0

    def __len__(self):
        return self.n // self.bs if self.drop_last else -(-self.n // self.bs)

    def __iter__(self):
        base = self.seed if self.seed is not None else int(torch.randint(0, 2**31 - 1, ()).item())
        rng = np.random.default_rng([base, self.epoch])
        self.epoch += 1
        perm = rng.permutation(self.n)
        for start in range(0, len(self) * self.bs, self.bs):
            idx = perm[start:start + self.bs]
            labels = rng.permutation(np.resize(self.classes, len(idx)))
            yield [(int(i), int(l)) for i, l in zip(idx, labels)]


def default_num_workers() -> int:
    return max(0, min(4, (os.cpu_count() or 2)))


def make_train_loader(images_u8, batch_size=32, classes=(0, 1, 2, 3), balanced=False,
                      num_workers=None, seed=None, pin_memory=True, drop_last=True):
    num_workers = default_num_workers() if num_workers is None else num_workers
    ds = RuntimeCorruptionDataset(images_u8, classes=classes, seed=seed)
    kw = dict(num_workers=num_workers, pin_memory=pin_memory and torch.cuda.is_available(),
              persistent_workers=num_workers > 0)
    if balanced:
        bs = BalancedCorruptionBatchSampler(len(ds), batch_size, classes, drop_last=drop_last, seed=seed)
        return DataLoader(ds, batch_sampler=bs, **kw)
    return DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=drop_last, **kw)


def make_eval_loader(images_u8, name_to_row, records, batch_size=64, classes=None,
                     return_index=False, num_workers=None, pin_memory=True):
    num_workers = default_num_workers() if num_workers is None else num_workers
    ds = ManifestCorruptionDataset(images_u8, name_to_row, records, classes=classes, return_index=return_index)
    # Not persistent: eval workers exit after each pass, so idle loaders don't hold RAM
    # (each worker gradually copies the Python objects it touches — "copy-on-read").
    return DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=num_workers,
                      pin_memory=pin_memory and torch.cuda.is_available(), persistent_workers=False)


class PetData:
    """Everything a Task 1–3 notebook needs, built by prepare_data()."""

    def __init__(self, **kw):
        self.__dict__.update(kw)

    def load_test_images(self, cache_dir=None):
        """FINAL EVALUATION ONLY: decode the quarantined official test images (cached)."""
        cache = Path(cache_dir or self.cache_dir) / "test_128.npz"
        out = load_clean_images(self.test_names, self.images_dir, cache_path=cache)
        return out["images"], {n: i for i, n in enumerate(out["names"])}, out["failed"]

    def summary(self) -> dict:
        return {"source": self.source, "images_dir": str(self.images_dir), "n_train": len(self.train_images),
                "n_val": len(self.val_images), "n_test": len(self.test_names),
                "n_val_records": len(self.val_manifest["records"]),
                "n_test_records": len(self.test_manifest["records"]),
                "split_sha256": self.hashes.get("split"), "val_manifest_sha256": self.hashes.get("val"),
                "test_manifest_sha256": self.hashes.get("test")}


def find_prep_outputs(search_root="/kaggle/input") -> Path | None:
    """Locate an attached output of notebook 00 (folder containing manifests/val_manifest.json)."""
    for p in sorted(Path(search_root).rglob("val_manifest.json")):
        root = p.parent.parent
        if (root / "manifests" / "test_manifest.json").exists() and (root / "artifacts" / "split_seed42.json").exists():
            return root
    return None


def prepare_data(input_root="/kaggle/input", work_dir="/kaggle/working", images_dir=None, annot_dir=None,
                 seed: int = SPLIT_SEED, verbose: bool = True) -> PetData:
    """Return split, clean train/val arrays and both manifests.

    1. If notebook 00's output is attached under `input_root`, its split + manifests are reused
       (guarantees byte-identical data across Tasks 1–3).
    2. Otherwise everything is regenerated deterministically (same seeds -> same files/hashes)
       and saved under `work_dir`.
    The official Oxford-IIIT Pet dataset must be attached in both cases (for the pixels).
    """
    input_root, work_dir = Path(input_root), Path(work_dir)
    cache_dir = work_dir / "cache"
    if images_dir is None or annot_dir is None:
        found = find_pet_dataset(input_root)
        images_dir = Path(images_dir or found["images_dir"])
        annot_dir = Path(annot_dir or found["annotations_dir"])
    test_names = read_split_list(Path(annot_dir) / "test.txt")

    prep = find_prep_outputs(input_root)
    if prep is not None:
        source = f"attached notebook-00 output: {prep}"
        split = load_json(prep / "artifacts" / "split_seed42.json")
        val_manifest = load_json(prep / "manifests" / "val_manifest.json")
        test_manifest = load_json(prep / "manifests" / "test_manifest.json")
        hashes = {"split": sha256_file(prep / "artifacts" / "split_seed42.json"),
                  "val": sha256_file(prep / "manifests" / "val_manifest.json"),
                  "test": sha256_file(prep / "manifests" / "test_manifest.json")}
        wanted = split["train"] + split["val"]
        cached = prep / "cache" / "trainval_128.npz"
        tv = None
        if cached.exists():
            z = load_clean_images(read_split_list(Path(annot_dir) / "trainval.txt"), images_dir,
                                  cache_path=cached, verbose=False)
            tv = z if set(wanted) <= set(z["names"]) else None
        if tv is None:
            tv = load_clean_images(wanted, images_dir, cache_path=cache_dir / "trainval_128.npz", verbose=verbose)
    else:
        source = "regenerated deterministically (notebook-00 output not attached)"
        trainval_names = read_split_list(Path(annot_dir) / "trainval.txt")
        tv = load_clean_images(trainval_names, images_dir, cache_path=cache_dir / "trainval_128.npz", verbose=verbose)
        split = make_split(tv["names"], VAL_FRACTION, seed)
        split["excluded_unreadable"] = tv["failed"]
        val_manifest = build_val_manifest(split["val"], seed)
        test_manifest = build_test_manifest(test_names, seed)
        hashes = {"split": save_json(split, work_dir / "artifacts" / "split_seed42.json", indent=1),
                  "val": save_json(val_manifest, work_dir / "manifests" / "val_manifest.json"),
                  "test": save_json(test_manifest, work_dir / "manifests" / "test_manifest.json")}

    row = {n: i for i, n in enumerate(tv["names"])}
    train_images = tv["images"][[row[n] for n in split["train"]]]
    val_images = tv["images"][[row[n] for n in split["val"]]]
    assert not set(split["train"]) & set(split["val"]) and not (set(split["train"]) | set(split["val"])) & set(test_names)
    data = PetData(source=source, images_dir=images_dir, annot_dir=annot_dir, cache_dir=cache_dir, split=split,
                   train_images=train_images, val_images=val_images,
                   val_name_to_row={n: i for i, n in enumerate(split["val"])},
                   val_manifest=val_manifest, test_manifest=test_manifest, test_names=test_names, hashes=hashes)
    if verbose:
        for k, v in data.summary().items():
            print(f"  {k:22s} {v}")
    return data


def seed_everything(seed: int = SPLIT_SEED):
    import random
    random.seed(seed)
    np.random.seed(seed)
    if torch is not None:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

In [ ]:
%%writefile restoration.py
"""
restoration.py — shared restoration components (Task 1, reused by Task 2 specialists and Task 3).

Contents
  * SSIM (Gaussian window, Wang et al. 2004) + PSNR, per-image and batched, differentiable
  * L1 + SSIM loss:  L = alpha * L1 + (1 - alpha) * (1 - SSIM)
  * ConvAutoencoder: conv encoder -> compressed spatial latent (1x1-conv bottleneck + dropout)
                     -> conv decoder, with optional LIMITED skip connections (deepest levels only)
  * train_one_epoch / evaluate (AMP-aware, per-corruption metrics)
  * ONNX export, PyTorch-vs-ONNX parity check, latency measurement
  * plotting helpers for restoration grids (target / input / output / |error|)
"""
from __future__ import annotations

import math
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

CLASS_NAMES = ("clean", "salt_pepper", "gaussian_blur", "occlusion")


# ======================================================================================
# Metrics
# ======================================================================================
def _gaussian_window(size: int, sigma: float, channels: int, device, dtype):
    x = torch.arange(size, device=device, dtype=dtype) - (size - 1) / 2
    g = torch.exp(-(x ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    w2d = g[:, None] * g[None, :]
    return w2d.expand(channels, 1, size, size).contiguous()


def ssim_map(x: torch.Tensor, y: torch.Tensor, window_size: int = 11, sigma: float = 1.5,
             data_range: float = 1.0) -> torch.Tensor:
    """SSIM map (B,C,H',W') with an 11x11 Gaussian window (sigma 1.5), 'valid' region only.
    Matches skimage.structural_similarity(gaussian_weights=True, sigma=1.5,
    use_sample_covariance=False, data_range=1)."""
    x, y = x.float(), y.float()
    c = x.shape[1]
    w = _gaussian_window(window_size, sigma, c, x.device, x.dtype)
    c1, c2 = (0.01 * data_range) ** 2, (0.03 * data_range) ** 2
    mu_x = F.conv2d(x, w, groups=c)
    mu_y = F.conv2d(y, w, groups=c)
    sxx = F.conv2d(x * x, w, groups=c) - mu_x ** 2
    syy = F.conv2d(y * y, w, groups=c) - mu_y ** 2
    sxy = F.conv2d(x * y, w, groups=c) - mu_x * mu_y
    return ((2 * mu_x * mu_y + c1) * (2 * sxy + c2)) / ((mu_x ** 2 + mu_y ** 2 + c1) * (sxx + syy + c2))


def ssim_per_image(x, y, **kw) -> torch.Tensor:
    return ssim_map(x, y, **kw).mean(dim=(1, 2, 3))


def psnr_per_image(x, y, data_range: float = 1.0) -> torch.Tensor:
    mse = ((x.float() - y.float()) ** 2).mean(dim=(1, 2, 3)).clamp_min(1e-10)
    return 10 * torch.log10(data_range ** 2 / mse)


def mae_per_image(x, y) -> torch.Tensor:
    return (x.float() - y.float()).abs().mean(dim=(1, 2, 3))


def val_score(psnr_mean: float, ssim_mean: float) -> float:
    """Optuna objective (alpha-independent, so the study cannot 'win' by changing the loss scale):
    equal-weight combination of structural similarity and reconstruction fidelity (PSNR/40, capped)."""
    return 0.5 * float(ssim_mean) + 0.5 * min(float(psnr_mean) / 40.0, 1.0)


# ======================================================================================
# Loss
# ======================================================================================
class L1SSIMLoss(nn.Module):
    """alpha * L1 + (1 - alpha) * (1 - SSIM). Always computed in float32 (AMP-safe)."""

    def __init__(self, alpha: float = 0.8):
        super().__init__()
        self.alpha = float(alpha)

    def forward(self, pred, target):
        pred, target = pred.float(), target.float()
        l1 = F.l1_loss(pred, target)
        s = ssim_map(pred, target).mean()
        total = self.alpha * l1 + (1 - self.alpha) * (1 - s)
        return total, {"l1": l1.detach(), "ssim": s.detach()}


# ======================================================================================
# Model
# ======================================================================================
def norm(c: int) -> nn.GroupNorm:
    # GroupNorm instead of BatchNorm: no running statistics, so train- and eval-mode behave the same
    # at any batch size (Optuna samples batch sizes 16-64), and frozen experts in Task 3 cannot drift.
    return nn.GroupNorm(min(8, c), c)


def conv_block(cin: int, cout: int, stride: int = 1) -> nn.Sequential:
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, stride, 1, bias=False), norm(cout), nn.SiLU(inplace=True),
        nn.Conv2d(cout, cout, 3, 1, 1, bias=False), norm(cout), nn.SiLU(inplace=True),
    )


class ConvAutoencoder(nn.Module):
    """Convolutional denoising autoencoder with a genuine compressed code.

    Encoder: 128 -> 64 -> 32 -> 16 -> 8 (stride-2 conv blocks), channels grow base x (1,2,4,8,8).
    Latent:  1x1 conv to `latent_channels` at 8x8 (e.g. 16 ch -> 1,024 values), then code dropout.
    Decoder: bilinear upsample + conv blocks (no transposed-conv checkerboard artefacts), sigmoid output.

    Skip connections (skip_levels = 0, 1 or 2) are BOTTLENECKED: the encoder feature at the
    16x16 (and 32x32) level is squeezed by a 1x1 conv to `skip_channels` channels and passes the same
    code dropout before reaching the decoder. The information the decoder receives is therefore
    bounded by the total code size

        code_dim = latent_dim + sum_over_skipped_levels(skip_channels * H_l * W_l),

    which is what `compression_ratio` reports (input values / code values). Full- and half-resolution
    features are never skipped. skip_channels=None reproduces the original full-width skips
    (e.g. 384 x 16 x 16 = 98,304 values, larger than the image); it is kept only for the ablation,
    which showed that such skips let the decoder ignore the latent.
    """

    def __init__(self, base_channels: int = 32, latent_channels: int = 32, dropout: float = 0.1,
                 skip_levels: int = 0, skip_channels: int | None = 4, depth: int = 4, max_mult: int = 8,
                 img_size: int = 128):
        super().__init__()
        assert 0 <= skip_levels <= 2, "only the two deepest levels may be skipped"
        self.config = dict(base_channels=base_channels, latent_channels=latent_channels, dropout=dropout,
                           skip_levels=skip_levels, skip_channels=skip_channels, depth=depth,
                           max_mult=max_mult, img_size=img_size)
        chs = [min(base_channels * 2 ** i, base_channels * max_mult) for i in range(depth + 1)]
        self.chs, self.depth, self.skip_levels, self.img_size = chs, depth, skip_levels, img_size
        self.stem = conv_block(3, chs[0], 1)
        self.down = nn.ModuleList([conv_block(chs[i], chs[i + 1], 2) for i in range(depth)])
        self.to_latent = nn.Conv2d(chs[-1], latent_channels, 1)
        self.code_dropout = nn.Dropout(dropout)
        self.from_latent = nn.Sequential(nn.Conv2d(latent_channels, chs[-1], 1, bias=False),
                                         norm(chs[-1]), nn.SiLU(inplace=True))
        # decoder stage j upsamples to resolution level (depth - j - 1); skips only for j < skip_levels
        self.skip_proj = nn.ModuleList()
        self.skip_shapes = []
        self.up = nn.ModuleList()
        for j in range(depth):
            lvl = depth - j - 1
            extra = 0
            if j < skip_levels:
                width = chs[lvl] if skip_channels is None else skip_channels
                self.skip_proj.append(nn.Identity() if skip_channels is None else nn.Conv2d(chs[lvl], skip_channels, 1))
                res = img_size // 2 ** lvl
                self.skip_shapes.append((width, res, res))
                extra = width
            self.up.append(conv_block(chs[lvl + 1] + extra, chs[lvl], 1))
        self.head = nn.Conv2d(chs[0], 3, 3, 1, 1)
        self.latent_shape = (latent_channels, img_size // 2 ** depth, img_size // 2 ** depth)

    # ---- code-size accounting ------------------------------------------------------------
    @property
    def latent_dim(self) -> int:
        return int(np.prod(self.latent_shape))

    @property
    def skip_dim(self) -> int:
        return int(sum(np.prod(s) for s in self.skip_shapes))

    @property
    def code_dim(self) -> int:
        """Total number of values the decoder receives from the encoder (latent + skip codes)."""
        return self.latent_dim + self.skip_dim

    @property
    def compression_ratio(self) -> float:
        return 3 * self.img_size ** 2 / self.code_dim

    # ---- forward -------------------------------------------------------------------------
    def encode(self, x):
        """Returns (latent, skip_codes). skip_codes[j] feeds decoder stage j (deepest first)."""
        feats = []
        h = self.stem(x)
        for blk in self.down:
            feats.append(h)
            h = blk(h)
        skips = [proj(feats[self.depth - j - 1]) for j, proj in enumerate(self.skip_proj)]
        return self.to_latent(h), skips

    def decode(self, z, skips=None):
        h = self.from_latent(self.code_dropout(z))
        for j, blk in enumerate(self.up):
            h = F.interpolate(h, scale_factor=2, mode="bilinear", align_corners=False)
            if j < self.skip_levels:
                s = skips[j] if skips is not None else h.new_zeros(h.shape[0], *self.skip_shapes[j])
                h = torch.cat([h, self.code_dropout(s)], dim=1)
            h = blk(h)
        return torch.sigmoid(self.head(h))

    def forward(self, x):
        z, skips = self.encode(x)
        return self.decode(z, skips)


@torch.no_grad()
def code_dependence(model, loader, device, max_batches=None) -> dict:
    """How much the output depends on each part of the code (evaluated on a manifest loader).

    full:        normal forward pass
    zero_latent: latent replaced by zeros (skips intact)   -> large drop = the latent is used
    zero_skips:  skip codes replaced by zeros (latent intact) -> drop = what the skips contribute
    """
    model.eval()
    acc = {k: {"psnr": [], "ssim": []} for k in ("full", "zero_latent", "zero_skips")}
    for b, batch in enumerate(loader):
        if max_batches is not None and b >= max_batches:
            break
        x, y = batch[0].to(device), batch[1].to(device)
        z, skips = model.encode(x)
        outs = {"full": model.decode(z, skips), "zero_latent": model.decode(torch.zeros_like(z), skips)}
        if model.skip_levels:
            outs["zero_skips"] = model.decode(z, [torch.zeros_like(s) for s in skips])
        for k, o in outs.items():
            o = o.float().clamp(0, 1)
            acc[k]["psnr"].append(psnr_per_image(o, y).cpu())
            acc[k]["ssim"].append(ssim_per_image(o, y).cpu())
    out = {}
    for k, d in acc.items():
        if d["psnr"]:
            out[f"{k}_psnr"] = torch.cat(d["psnr"]).mean().item()
            out[f"{k}_ssim"] = torch.cat(d["ssim"]).mean().item()
    out["latent_drop_db"] = out["full_psnr"] - out["zero_latent_psnr"]
    if "zero_skips_psnr" in out:
        out["skip_drop_db"] = out["full_psnr"] - out["zero_skips_psnr"]
    return out


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ======================================================================================
# Training / evaluation
# ======================================================================================
def make_optimizer_and_scheduler(model, lr, weight_decay, steps_per_epoch, epochs, warmup_epochs=1):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    total = max(1, steps_per_epoch * epochs)
    warm = max(1, int(steps_per_epoch * warmup_epochs))

    def lr_lambda(step):
        if step < warm:
            return (step + 1) / warm
        t = (step - warm) / max(1, total - warm)
        return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * min(1.0, t)))

    return opt, torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)


def make_scaler(device, amp=True):
    """One GradScaler per training run (fp16 AMP on CUDA only; a no-op on CPU)."""
    return torch.amp.GradScaler("cuda", enabled=bool(amp and device.type == "cuda"))


def train_one_epoch(model, loader, loss_fn, optimizer, scheduler, device, scaler=None, amp=True,
                    grad_clip=1.0, max_batches=None) -> dict:
    model.train()
    use_amp = amp and device.type == "cuda"
    if scaler is None:
        scaler = make_scaler(device, amp)
    tot = {"loss": 0.0, "l1": 0.0, "ssim": 0.0}
    n = 0
    for b, batch in enumerate(loader):
        if max_batches is not None and b >= max_batches:
            break
        x, y = batch[0].to(device, non_blocking=True), batch[1].to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            pred = model(x)
        loss, parts = loss_fn(pred, y)
        if not torch.isfinite(loss):
            raise FloatingPointError(f"non-finite loss at batch {b}")
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        if grad_clip:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(optimizer)
        scaler.update()
        if scheduler is not None:
            scheduler.step()
        bs = x.shape[0]
        tot["loss"] += loss.item() * bs
        tot["l1"] += parts["l1"].item() * bs
        tot["ssim"] += parts["ssim"].item() * bs
        n += bs
    return {k: v / max(n, 1) for k, v in tot.items()}


@torch.no_grad()
def evaluate(model, loader, device, loss_fn=None, amp=True, per_image=False, include_input=False) -> dict:
    """Metrics over a (manifest) loader. Returns means overall and per corruption type.
    per_image=True also returns numpy arrays aligned with the dataset order.
    include_input=True also scores the corrupted input itself (the 'do nothing' baseline)."""
    model.eval()
    use_amp = amp and device.type == "cuda"
    keys = ["psnr", "ssim", "mae"] + (["in_psnr", "in_ssim", "in_mae"] if include_input else [])
    acc = {k: [] for k in keys}
    labels, losses = [], []
    for batch in loader:
        x, y, lab = batch[0].to(device), batch[1].to(device), batch[2]
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            pred = model(x)
        pred = pred.float().clamp(0, 1)
        if loss_fn is not None:
            losses.append(loss_fn(pred, y)[0].item() * x.shape[0])
        acc["psnr"].append(psnr_per_image(pred, y).cpu())
        acc["ssim"].append(ssim_per_image(pred, y).cpu())
        acc["mae"].append(mae_per_image(pred, y).cpu())
        if include_input:
            acc["in_psnr"].append(psnr_per_image(x, y).cpu())
            acc["in_ssim"].append(ssim_per_image(x, y).cpu())
            acc["in_mae"].append(mae_per_image(x, y).cpu())
        labels.append(lab.cpu())
    arr = {k: torch.cat(v).numpy() for k, v in acc.items()}
    labels = torch.cat(labels).numpy()
    out = {k: float(v.mean()) for k, v in arr.items()}
    for c, name in enumerate(CLASS_NAMES):
        m = labels == c
        if m.any():
            for k, v in arr.items():
                out[f"{k}/{name}"] = float(v[m].mean())
    out["score"] = val_score(out["psnr"], out["ssim"])
    if loss_fn is not None:
        out["loss"] = sum(losses) / len(labels)
    if per_image:
        out["per_image"] = {**arr, "label": labels}
    return out


@torch.no_grad()
def predict_numpy(model, imgs_hwc: np.ndarray, device, batch_size=64) -> np.ndarray:
    """(N,H,W,3) float [0,1] -> restored (N,H,W,3) float [0,1]."""
    model.eval()
    outs = []
    for i in range(0, len(imgs_hwc), batch_size):
        x = torch.from_numpy(np.ascontiguousarray(imgs_hwc[i:i + batch_size].transpose(0, 3, 1, 2))).float().to(device)
        outs.append(model(x).float().clamp(0, 1).cpu().numpy().transpose(0, 2, 3, 1))
    return np.concatenate(outs)


# ======================================================================================
# ONNX
# ======================================================================================
def export_onnx(model, path, img_size=128, opset=17, input_name="input", output_name="output"):
    model = model.eval().cpu()
    dummy = torch.rand(1, 3, img_size, img_size)
    kw = dict(input_names=[input_name], output_names=[output_name], opset_version=opset,
              dynamic_axes={input_name: {0: "batch"}, output_name: {0: "batch"}})
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with torch.no_grad():
        try:
            torch.onnx.export(model, dummy, str(path), dynamo=False, **kw)
        except TypeError:          # torch < 2.5 has no `dynamo` kwarg
            torch.onnx.export(model, dummy, str(path), **kw)
    try:
        import onnx
        onnx.checker.check_model(onnx.load(str(path)))
    except ImportError:
        pass
    return path


def verify_onnx(model, path, x: torch.Tensor, atol=1e-4) -> dict:
    """Compare PyTorch (CPU, eval) and onnxruntime outputs on the same batch."""
    import onnxruntime as ort
    model = model.eval().cpu()
    sess = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    with torch.no_grad():
        ref = model(x.cpu().float()).numpy()
    out = sess.run(None, {sess.get_inputs()[0].name: x.cpu().float().numpy()})[0]
    diff = np.abs(out - ref)
    p_ref = psnr_per_image(torch.from_numpy(ref), torch.from_numpy(out))
    return {"max_abs_diff": float(diff.max()), "mean_abs_diff": float(diff.mean()),
            "min_psnr_torch_vs_onnx": float(p_ref.min()), "n_images": int(len(x)),
            "passed": bool(diff.max() < atol)}


def onnx_latency_ms(path, img_size=128, batch=1, n_warmup=10, n_runs=100) -> dict:
    import onnxruntime as ort
    sess = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    name = sess.get_inputs()[0].name
    x = np.random.rand(batch, 3, img_size, img_size).astype(np.float32)
    for _ in range(n_warmup):
        sess.run(None, {name: x})
    ts = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        sess.run(None, {name: x})
        ts.append((time.perf_counter() - t0) * 1000)
    return {"median_ms": float(np.median(ts)), "p90_ms": float(np.percentile(ts, 90)), "batch": batch}


# ======================================================================================
# Plotting
# ======================================================================================
def restoration_grid(rows: list[dict], path=None, title=None, err_vmax=0.5):
    """rows: dicts with 'clean', 'corrupted', 'restored' (HxWx3 float [0,1]) and optional 'label'.
    Columns: clean target | corrupted input | reconstruction | absolute error map."""
    import matplotlib.pyplot as plt
    n = len(rows)
    fig, axes = plt.subplots(n, 4, figsize=(9.2, 2.35 * n), squeeze=False)
    for i, r in enumerate(rows):
        err = np.abs(r["restored"] - r["clean"]).mean(-1)
        panels = [r["clean"], r["corrupted"], r["restored"]]
        for j, im in enumerate(panels):
            axes[i, j].imshow(np.clip(im, 0, 1))
        hm = axes[i, 3].imshow(err, cmap="magma", vmin=0, vmax=err_vmax)
        for ax in axes[i]:
            ax.set_xticks([]); ax.set_yticks([])
        axes[i, 0].set_ylabel(r.get("label", ""), fontsize=7.5, rotation=0, ha="right", va="center", labelpad=4)
        if "in_text" in r:
            axes[i, 1].set_xlabel(r["in_text"], fontsize=7.5)
        if "out_text" in r:
            axes[i, 2].set_xlabel(r["out_text"], fontsize=7.5)
    for j, t in enumerate(["clean target", "corrupted input", "reconstruction", "|error| (mean RGB)"]):
        axes[0, j].set_title(t, fontsize=9)
    fig.colorbar(hm, ax=axes[:, 3].tolist(), fraction=0.046, pad=0.02, shrink=0.6)
    if title:
        fig.suptitle(title, fontsize=11)
    if path:
        fig.savefig(path, dpi=150, bbox_inches="tight")
    return fig

In [ ]:
%%writefile classifier.py
"""
classifier.py — corruption classifier for Task 2 (reused as the gating network in Task 3).

Classes (same order everywhere, incl. the Task 3 gate weights w0..w3):
    0 = clean, 1 = salt_pepper, 2 = gaussian_blur, 3 = occlusion

Design notes
  * The first conv block runs at FULL resolution (128x128): salt-and-pepper noise and slight blur are
    pixel-level cues that are destroyed by early downsampling.
  * GroupNorm (no running statistics): train/eval behave identically, which matters when the network is
    later fine-tuned as the Task 3 gate with small batches.
  * Global AVERAGE + global MAX pooling are concatenated: max pooling responds to rare extreme activations
    (isolated black/white impulses, hard edges of black occlusion boxes); average pooling to global
    statistics (overall loss of high frequencies under blur).
  * forward() returns LOGITS; softmax / temperature are applied by the caller (Task 3: softmax(G(x)/tau)).
"""
from __future__ import annotations

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

CLASS_NAMES = ("clean", "salt_pepper", "gaussian_blur", "occlusion")
CHANNEL_CONFIGS = {
    "16-32-64-128": (16, 32, 64, 128),
    "32-64-128-256": (32, 64, 128, 256),
    "32-64-128-256-256": (32, 64, 128, 256, 256),
}


def _gn(c):
    return nn.GroupNorm(min(8, c), c)


class CorruptionClassifier(nn.Module):
    def __init__(self, channels=(32, 64, 128, 256), dropout: float = 0.2, num_classes: int = 4):
        super().__init__()
        if isinstance(channels, str):
            channels = CHANNEL_CONFIGS[channels]
        channels = tuple(int(c) for c in channels)
        self.config = dict(channels=list(channels), dropout=dropout, num_classes=num_classes)
        layers, cin = [], 3
        for i, c in enumerate(channels):
            stride = 1 if i == 0 else 2            # stage 0 keeps full resolution
            layers += [nn.Conv2d(cin, c, 3, stride, 1, bias=False), _gn(c), nn.SiLU(inplace=True),
                       nn.Conv2d(c, c, 3, 1, 1, bias=False), _gn(c), nn.SiLU(inplace=True)]
            cin = c
        self.features = nn.Sequential(*layers)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(2 * cin, num_classes)

    def forward(self, x):
        h = self.features(x)
        h = torch.cat([h.mean(dim=(2, 3)), h.amax(dim=(2, 3))], dim=1)
        return self.fc(self.dropout(h))


class ClassifierWithProbs(nn.Module):
    """ONNX wrapper: returns (logits, probabilities)."""

    def __init__(self, clf):
        super().__init__()
        self.clf = clf

    def forward(self, x):
        logits = self.clf(x)
        return logits, torch.softmax(logits, dim=1)


# --------------------------------------------------------------------------------------
# Training / evaluation
# --------------------------------------------------------------------------------------
def train_one_epoch(model, loader, optimizer, scheduler, device, scaler, amp=True, grad_clip=1.0, label_smoothing=0.0):
    model.train()
    use_amp = amp and device.type == "cuda"
    tot_loss, correct, n = 0.0, 0, 0
    for x, _, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = model(x)
        loss = F.cross_entropy(logits.float(), y, label_smoothing=label_smoothing)
        if not torch.isfinite(loss):
            raise FloatingPointError("non-finite classifier loss")
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        if grad_clip:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(optimizer)
        scaler.update()
        if scheduler is not None:
            scheduler.step()
        tot_loss += loss.item() * len(y)
        correct += (logits.argmax(1) == y).sum().item()
        n += len(y)
    return {"loss": tot_loss / max(n, 1), "acc": correct / max(n, 1)}


@torch.no_grad()
def predict(model, loader, device, amp=True):
    """Returns (labels, probs[N,4], logits[N,4]) over a manifest loader (dataset order)."""
    model.eval()
    use_amp = amp and device.type == "cuda"
    labels, logits = [], []
    for batch in loader:
        x = batch[0].to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            lg = model(x)
        logits.append(lg.float().cpu())
        labels.append(batch[2])
    logits = torch.cat(logits)
    return torch.cat(labels).numpy(), torch.softmax(logits, 1).numpy(), logits.numpy()


def classification_metrics(y_true, probs) -> dict:
    """Accuracy, macro P/R/F1, per-class P/R/F1/support, row-normalised confusion matrix, CE loss."""
    from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
    y_pred = probs.argmax(1)
    labels = list(range(len(CLASS_NAMES)))
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    mp, mr, mf, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_norm = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    ce = float(-np.log(np.clip(probs[np.arange(len(y_true)), y_true], 1e-12, 1)).mean())
    out = {"acc": float(accuracy_score(y_true, y_pred)), "macro_precision": float(mp), "macro_recall": float(mr),
           "macro_f1": float(mf), "loss": ce, "cm": cm, "cm_norm": cm_norm}
    for i, n in enumerate(CLASS_NAMES):
        out[f"precision/{n}"], out[f"recall/{n}"], out[f"f1/{n}"], out[f"support/{n}"] = float(p[i]), float(r[i]), float(f[i]), int(s[i])
    return out


def per_class_table(m: dict):
    import pandas as pd
    rows = [{"class": n, "precision": m[f"precision/{n}"], "recall": m[f"recall/{n}"], "f1": m[f"f1/{n}"],
             "support": m[f"support/{n}"]} for n in CLASS_NAMES]
    rows.append({"class": "macro avg", "precision": m["macro_precision"], "recall": m["macro_recall"],
                 "f1": m["macro_f1"], "support": int(sum(m[f"support/{n}"] for n in CLASS_NAMES))})
    rows.append({"class": "accuracy", "precision": np.nan, "recall": np.nan, "f1": m["acc"],
                 "support": int(sum(m[f"support/{n}"] for n in CLASS_NAMES))})
    return pd.DataFrame(rows).set_index("class")


def plot_confusion(cm_norm, cm_counts=None, title="", path=None, ax=None):
    import matplotlib.pyplot as plt
    own = ax is None
    if own:
        fig, ax = plt.subplots(figsize=(4.8, 4.2))
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    short = ["clean", "s&p", "blur", "occl."]
    ax.set_xticks(range(4)); ax.set_xticklabels(short); ax.set_yticks(range(4)); ax.set_yticklabels(short)
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
    for i in range(4):
        for j in range(4):
            txt = f"{cm_norm[i, j]:.2f}" + (f"\n({cm_counts[i, j]})" if cm_counts is not None else "")
            ax.text(j, i, txt, ha="center", va="center", fontsize=8,
                    color="white" if cm_norm[i, j] > 0.6 else "black")
    ax.set_title(title, fontsize=10)
    if own:
        fig.colorbar(im, ax=ax, fraction=0.046)
        fig.tight_layout()
        if path:
            fig.savefig(path, dpi=150)
        return fig
    return im

In [ ]:
import os, sys, json, time, math, shutil, importlib, warnings, gc
from pathlib import Path
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import optuna

sys.path.insert(0, str(Path.cwd()))
import pet_data as pdt, restoration as R, classifier as C
importlib.reload(pdt); importlib.reload(R); importlib.reload(C)
warnings.filterwarnings("ignore", category=optuna.exceptions.ExperimentalWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ============================ MAIN SWITCH ============================
MODE = "smoke"          # "smoke" first, then "full"
# =====================================================================
MODE = os.environ.get("T2_MODE", MODE)

SETTINGS = {
    "smoke": dict(train_subset=512, val_per_class=32, test_images=40,
                  clf_trials=3, clf_trial_epochs=2, clf_timeout=None, clf_final_epochs=3, clf_patience=99,
                  spec_trials=2, spec_trial_epochs=1, spec_timeout=None, spec_final_epochs=2, spec_patience=99,
                  sample_every=1, overfit_steps=100),
    "full":  dict(train_subset=None, val_per_class=None, test_images=None,
                  clf_trials=25, clf_trial_epochs=8, clf_timeout=3600, clf_final_epochs=40, clf_patience=10,
                  spec_trials=20, spec_trial_epochs=5, spec_timeout=int(2.5 * 3600), spec_final_epochs=60, spec_patience=12,
                  sample_every=5, overfit_steps=100),
}
S = dict(SETTINGS[MODE])
S.update(json.loads(os.environ.get("T2_SETTINGS_OVERRIDE", "{}")))   # local CPU testing only

IS_KAGGLE  = Path("/kaggle/input").exists()
INPUT_ROOT = Path(os.environ.get("PET_INPUT_ROOT", "/kaggle/input"))
WORK_DIR   = Path(os.environ.get("WORK_DIR", "/kaggle/working" if IS_KAGGLE else "./work")).resolve()
IMAGES_DIR, ANNOT_DIR = None, None

TAG = "task2" if MODE == "full" else "task2_smoke"
DIRS = {k: WORK_DIR / k / TAG for k in ["checkpoints", "models", "figures", "results", "optuna"]}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
AMP = True
NUM_WORKERS = pdt.default_num_workers()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pdt.seed_everything(SEED)
EXPERTS = {1: "salt_pepper", 2: "gaussian_blur", 3: "occlusion"}     # label -> specialist name (0 = identity)
SPEC_ORDER = [3, 2, 1]        # hardest first: early pruning in the shared study is decided on occlusion
MAX_CODE = 3 * 128 * 128 // 8  # same genuine-bottleneck rule as Task 1 v2 (<= 6,144 values)
SPEC_WEIGHT_DECAY = 1e-4

USE_WANDB = True
WANDB_PROJECT = "genai-assignment1"
WANDB_ENTITY = None
WANDB_SECRET_NAME = "WANDB_API_KEY"
GROUP = "task2" if MODE == "full" else "task2-smoke"

print(f"MODE={MODE} | device={device} {torch.cuda.get_device_name(0) if device.type == 'cuda' else ''} | "
      f"torch {torch.__version__} | optuna {optuna.__version__} | workers={NUM_WORKERS}")
print(json.dumps(S))

In [ ]:
WANDB_OK = False
if USE_WANDB:
    try:
        import wandb
        if IS_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            wandb.login(key=UserSecretsClient().get_secret(WANDB_SECRET_NAME))
        WANDB_OK = True
    except Exception as e:
        print("W&B login error:", repr(e))

def wb_init(name, job_type, config=None, **kw):
    if not WANDB_OK:
        return None
    if wandb.run is not None:
        wandb.finish()
    return wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, group=GROUP, name=name, job_type=job_type,
                      tags=["task2", MODE], config=config or {}, **kw)

def wb_log_figs(run, figs: dict):
    if run is not None:
        run.log({k: wandb.Image(str(v)) for k, v in figs.items()})

print("W&B:", "on" if WANDB_OK else "off")
if USE_WANDB and not WANDB_OK:
    raise RuntimeError("W&B login failed — attach the WANDB_API_KEY secret to THIS notebook (Add-ons -> Secrets) "
                       "and turn Internet on. Nothing has been trained.")

## 1. Data

In [ ]:
data = pdt.prepare_data(INPUT_ROOT, WORK_DIR, IMAGES_DIR, ANNOT_DIR)
train_images = data.train_images if S["train_subset"] is None else data.train_images[:S["train_subset"]]
val_records = data.val_manifest["records"]
if S["val_per_class"] is not None:
    val_records = [r for c in range(4) for r in [x for x in val_records if x["label"] == c][:S["val_per_class"]]]
val_loader = pdt.make_eval_loader(data.val_images, data.val_name_to_row, val_records, batch_size=128, num_workers=NUM_WORKERS)
val_loader_by_type = {k: pdt.make_eval_loader(data.val_images, data.val_name_to_row, val_records, batch_size=128,
                                              classes=(k,), num_workers=NUM_WORKERS) for k in EXPERTS}
print(f"\ntrain images {len(train_images)} | val records {len(val_records)} "
      f"({pd.Series([pdt.CLASS_NAMES[r['label']] for r in val_records]).value_counts().to_dict()})")

# Optional: the Task 1 universal model for the universal-vs-routed comparison
T1_CKPT = None
for p in sorted(INPUT_ROOT.rglob("task1_best.pth")):
    if p.parent.name == "task1":
        T1_CKPT = p
print("Task 1 checkpoint:", T1_CKPT)

### Pre-flight gate
Nothing trains unless these pass.

In [ ]:
pre = []
def pf(name, ok, detail="", fatal=True):
    pre.append({"check": name, "passed": bool(ok), "fatal": fatal, "detail": str(detail)})
    print(("PASS " if ok else ("FAIL " if fatal else "WARN ")) + name + (f"  — {detail}" if detail else ""))

gpu = torch.cuda.is_available()
pf("GPU available", gpu, torch.cuda.get_device_name(0) if gpu else "Settings -> Accelerator -> GPU T4", fatal=(MODE == "full"))
wb_ok, wb_detail = False, "USE_WANDB is False"
if USE_WANDB:
    try:
        r = wb_init("preflight", "preflight", {"mode": MODE, "settings": S})
        r.log({"preflight/ok": 1}); wb_detail = r.url; r.finish(); wb_ok = True
    except Exception as e:
        wb_detail = repr(e)
pf("W&B login + test run", wb_ok or not USE_WANDB, wb_detail)
pf("dataset found", len(data.split["train"]) + len(data.split["val"]) >= 3600 and len(data.test_names) >= 3600,
   f"{len(data.split['train'])} train / {len(data.split['val'])} val / {len(data.test_names)} test")
pf("notebook-00 output attached (same split + manifests as Tasks 1 and 3)", data.source.startswith("attached"), data.source, fatal=False)
pf("split / manifest hashes recorded", all(data.hashes.values()), {k: v[:12] for k, v in data.hashes.items()})
t1_ok = False
if T1_CKPT is not None:
    try:
        ck1 = torch.load(T1_CKPT, map_location="cpu", weights_only=False)
        R.ConvAutoencoder(**ck1["model_config"]).load_state_dict(ck1["model"]); t1_ok = True
    except Exception as e:
        T1_CKPT = None; print("   Task 1 checkpoint not loadable with the current architecture:", repr(e)[:120])
pf("Task 1 v2 model attached (universal-vs-routed comparison)", t1_ok, "optional", fatal=False)
try:
    import onnx, onnxruntime, sklearn
    pf("onnx + onnxruntime + sklearn importable", True, f"onnx {onnx.__version__}, ort {onnxruntime.__version__}, sklearn {sklearn.__version__}")
except Exception as e:
    pf("onnx + onnxruntime + sklearn importable", False, repr(e))
ok = True
for d in DIRS.values():
    try:
        (d / ".probe").write_text("x"); (d / ".probe").unlink()
    except Exception:
        ok = False
pf("output folders writeable", ok, str(WORK_DIR))
free = shutil.disk_usage(WORK_DIR).free / 1e9
pf("free disk > 5 GB", free > 5, f"{free:.1f} GB")
pre_df = pd.DataFrame(pre)
bad = pre_df[pre_df.fatal & ~pre_df.passed]
assert bad.empty, "PRE-FLIGHT FAILED — nothing has been trained. Fix these and re-run:\n" + bad.to_string()
print(f"\nPRE-FLIGHT OK ({int(pre_df.passed.sum())}/{len(pre_df)} passed) — safe to train.")

In [ ]:
def records_to_arrays(records, images, name_to_row):
    clean = np.stack([images[name_to_row[r["image"]]].astype(np.float32) / 255 for r in records])
    corrupted = np.stack([pdt.apply_corruption(c, r) for c, r in zip(clean, records)])
    return corrupted, clean

def export_onnx_multi(model, path, output_names):
    model = model.eval().cpu()
    kw = dict(input_names=["input"], output_names=output_names, opset_version=17,
              dynamic_axes={"input": {0: "batch"}, **{o: {0: "batch"} for o in output_names}})
    with torch.no_grad():
        try:
            torch.onnx.export(model, torch.rand(1, 3, 128, 128), str(path), dynamo=False, **kw)
        except TypeError:
            torch.onnx.export(model, torch.rand(1, 3, 128, 128), str(path), **kw)
    import onnx
    onnx.checker.check_model(onnx.load(str(path)))

## 2. Unit tests
These run in both modes and catch wiring bugs before any long training.

In [ ]:
tests = []
def check(name, ok, detail=""):
    tests.append({"test": name, "passed": bool(ok), "detail": str(detail)})
    print(("PASS " if ok else "FAIL ") + name + (f"  — {detail}" if detail else ""))

x = torch.rand(8, 3, 128, 128, device=device)
for cfg in C.CHANNEL_CONFIGS:
    m = C.CorruptionClassifier(cfg, 0.2).to(device).eval()
    with torch.no_grad():
        lg = m(x)
    check(f"classifier {cfg}: logits (8,4), finite", tuple(lg.shape) == (8, 4) and torch.isfinite(lg).all(), f"{R.count_params(m) / 1e6:.2f}M params")

bal = pdt.make_train_loader(train_images, batch_size=32, balanced=True, num_workers=0)
counts = [np.bincount(b[2].numpy(), minlength=4) for _, b in zip(range(5), bal)]
check("balanced batches: exactly 8 of each class in every batch of 32", all((c == 8).all() for c in counts), counts[0].tolist())
for k, n in EXPERTS.items():
    b = next(iter(pdt.make_train_loader(train_images, batch_size=16, classes=(k,), num_workers=0)))
    check(f"specialist loader '{n}' yields only label {k}", set(b[2].tolist()) == {k})

m = C.CorruptionClassifier("16-32-64-128", 0.0).to(device).train()
xb, _, yb = next(iter(bal)); xb, yb = xb.to(device), yb.to(device)
opt = torch.optim.Adam(m.parameters(), lr=2e-3)
losses = []
for _ in range(S["overfit_steps"]):
    loss = F.cross_entropy(m(xb), yb); opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
acc = (m(xb).argmax(1) == yb).float().mean().item()
# wiring check: the loss must fall well below chance level (ln 4 = 1.386) and accuracy must clearly beat chance (0.25)
check("classifier learns one balanced batch (loss < 60% of initial, acc >= 0.5)", losses[-1] < 0.6 * losses[0] and acc >= 0.5,
      f"loss {losses[0]:.3f} -> {losses[-1]:.3f}, acc {acc:.2f}")

mm = C.classification_metrics(np.array([0, 1, 2, 3, 0]), np.eye(4)[[0, 1, 2, 3, 1]])
check("metrics: accuracy/macro-F1/confusion computed correctly", abs(mm["acc"] - 0.8) < 1e-9 and mm["cm"][0, 1] == 1)

sp = R.ConvAutoencoder(32, 16, 0.1, 2, 4).to(device).eval()
with torch.no_grad():
    y = sp(x)
check("specialist AE: output shape + 8x code rule", tuple(y.shape) == (8, 3, 128, 128) and sp.code_dim <= MAX_CODE, f"code {sp.code_dim}")

probe = DIRS["models"] / "_probe.onnx"
export_onnx_multi(C.ClassifierWithProbs(C.CorruptionClassifier("16-32-64-128")), probe, ["logits", "probs"])
import onnxruntime as ort
xx = torch.rand(3, 3, 128, 128)
o = ort.InferenceSession(str(probe), providers=["CPUExecutionProvider"]).run(None, {"input": xx.numpy()})
check("classifier ONNX has logits + probs; probs sum to 1", len(o) == 2 and np.allclose(o[1].sum(1), 1, atol=1e-5))
probe.unlink()
del m, sp, opt, bal

tests_df = pd.DataFrame(tests)
assert tests_df.passed.all(), f"UNIT TESTS FAILED:\n{tests_df[~tests_df.passed]}"
print(f"\nall {len(tests_df)} tests passed")

## 3. Corruption classifier
* **Loss:** multiclass cross-entropy. **Batches:** exactly balanced (B/4 samples of each condition per batch, via `BalancedCorruptionBatchSampler`). Corruptions are sampled at runtime.
* **Optuna objective:** validation **macro-F1** on the fixed validation manifest. **Search space:** lr, batch size, convolutional channel configuration, dropout, weight decay.
* **Sampler:** TPE (seed 42). **Pruner:** median pruner. **Storage:** SQLite.

In [ ]:
CLF_SPACE = {
    "lr":           {"type": "float", "low": 1e-4, "high": 3e-3, "log": True},
    "batch_size":   {"type": "categorical", "choices": [32, 64, 128]},        # multiples of 4 -> exactly balanced
    "channels":     {"type": "categorical", "choices": list(C.CHANNEL_CONFIGS)},
    "dropout":      {"type": "float", "low": 0.0, "high": 0.5},
    "weight_decay": {"type": "float", "low": 1e-6, "high": 1e-2, "log": True},
}
CLF_BASELINE = {"lr": 1e-3, "batch_size": 64, "channels": "32-64-128-256", "dropout": 0.2, "weight_decay": 1e-4}

def suggest(trial, space):
    hp = {}
    for k, s in space.items():
        hp[k] = (trial.suggest_float(k, s["low"], s["high"], log=s.get("log", False)) if s["type"] == "float"
                 else trial.suggest_categorical(k, s["choices"]))
    return hp

def train_classifier(hp, epochs, run=None, trial=None, ckpt=None, patience=None):
    torch.manual_seed(SEED + (trial.number if trial else 0))
    model = C.CorruptionClassifier(hp["channels"], hp["dropout"]).to(device)
    loader = pdt.make_train_loader(train_images, batch_size=hp["batch_size"], balanced=True, num_workers=NUM_WORKERS)
    opt, sch = R.make_optimizer_and_scheduler(model, hp["lr"], hp["weight_decay"], len(loader), epochs)
    scaler = R.make_scaler(device, AMP)
    best, best_ep, hist = -1.0, -1, []
    try:
        for ep in range(epochs):
            t0 = time.time()
            tr = C.train_one_epoch(model, loader, opt, sch, device, scaler, AMP)
            yv, pv, _ = C.predict(model, val_loader, device, AMP)
            mv = C.classification_metrics(yv, pv)
            rec = {"epoch": ep, "lr": sch.get_last_lr()[0], "time_s": time.time() - t0, "train/loss": tr["loss"], "train/acc": tr["acc"],
                   "val/loss": mv["loss"], "val/acc": mv["acc"], "val/macro_f1": mv["macro_f1"],
                   **{f"val/recall/{n}": mv[f"recall/{n}"] for n in C.CLASS_NAMES}}
            hist.append(rec)
            if run:
                run.log(rec)
            if mv["macro_f1"] > best:
                best, best_ep = mv["macro_f1"], ep
                if trial:
                    trial.set_user_attr("val_acc", mv["acc"]); trial.set_user_attr("best_epoch", ep)
                if ckpt:
                    torch.save({"model": model.state_dict(), "model_config": model.config, "hp": hp, "epoch": ep,
                                "val_macro_f1": best}, ckpt)
            if not trial:
                print(f"ep {ep:3d} | train loss {tr['loss']:.4f} acc {tr['acc']:.3f} | val loss {mv['loss']:.4f} "
                      f"acc {mv['acc']:.3f} macro-F1 {mv['macro_f1']:.4f}{' *' if best_ep == ep else ''} | {rec['time_s']:.0f}s")
            if trial:
                trial.report(mv["macro_f1"], ep)
                if trial.should_prune():
                    raise optuna.TrialPruned()
            if patience and ep - best_ep >= patience:
                print(f"early stopping (no improvement for {patience} epochs)"); break
    finally:
        del loader
    return model, best, hist

def clf_objective(trial):
    hp = suggest(trial, CLF_SPACE)
    run = wb_init(f"clf-trial-{trial.number:03d}", "clf-trial", {**hp, "trial": trial.number})
    state = "complete"
    try:
        _, best, _ = train_classifier(hp, S["clf_trial_epochs"], run, trial)
    except optuna.TrialPruned:
        state = "pruned"; raise
    finally:
        if run:
            run.summary.update({"state": state}); run.finish()
        gc.collect(); torch.cuda.empty_cache()
    return best

def run_study(name, objective, n_trials, timeout, baseline, pruner):
    db = DIRS["optuna"] / f"{name}.db"
    if MODE == "smoke" and db.exists():
        db.unlink()
    st = optuna.create_study(study_name=name, storage=f"sqlite:///{db}", direction="maximize", load_if_exists=True,
                             sampler=optuna.samplers.TPESampler(seed=SEED, multivariate=True, n_startup_trials=min(8, n_trials)),
                             pruner=pruner)
    if len(st.trials) == 0:
        st.enqueue_trial(baseline)
    t0 = time.time()
    done = lambda: len([t for t in st.trials if "invalid" not in t.user_attrs and
                        t.state in (optuna.trial.TrialState.COMPLETE, optuna.trial.TrialState.PRUNED)])
    attempts = 0
    while done() < n_trials and attempts < 6 * n_trials:
        left = None if timeout is None else timeout - (time.time() - t0)
        if left is not None and left <= 0:
            print("time budget reached"); break
        st.optimize(objective, n_trials=1, timeout=left, gc_after_trial=True, catch=(FloatingPointError,))
        attempts += 1
    print(f"study '{name}': {done()} valid trials in {(time.time() - t0) / 60:.1f} min; best #{st.best_trial.number} = {st.best_value:.4f}")
    return st, db

def summarise_study(st, db, space, baseline, label):
    df = st.trials_dataframe(attrs=("number", "value", "state", "params", "user_attrs", "duration"))
    df.to_csv(DIRS["results"] / f"{label}_optuna_trials.csv", index=False)
    states = df["state"].value_counts().to_dict()
    if "user_attrs_invalid" in df:
        n_inv = int(df["user_attrs_invalid"].notna().sum())
        states["INVALID (rejected, not trained)"] = n_inv; states["PRUNED"] = states.get("PRUNED", 0) - n_inv
    display(df.sort_values("value", ascending=False).head(8))
    ps = optuna.create_study(direction="maximize"); ps.add_trials([t for t in st.trials if "invalid" not in t.user_attrs])
    figs = {}
    from optuna.visualization import matplotlib as ovm
    for nm, fn in [("history", ovm.plot_optimization_history), ("importance", ovm.plot_param_importances),
                   ("slice", ovm.plot_slice), ("parallel", ovm.plot_parallel_coordinate)]:
        try:
            obj = fn(ps)
            fig = (obj.flat[0] if hasattr(obj, "flat") else obj).figure
            fig.set_size_inches(12 if nm in ("slice", "parallel") else 7, 4.5)
            p = DIRS["figures"] / f"{label}_optuna_{nm}.png"
            fig.savefig(p, dpi=140, bbox_inches="tight"); plt.show(); figs[f"optuna/{label}_{nm}"] = p
        except Exception as e:
            print(f"plot {nm} skipped: {e}")
    plt.style.use("default")          # optuna's matplotlib backend switches to ggplot (grid lines over images)
    info = {"best_trial": st.best_trial.number, "best_value": st.best_value, "best_params": st.best_params,
            "user_attrs": st.best_trial.user_attrs, "n_trials": len(st.trials), "states": states,
            "search_space": space, "baseline": baseline}
    pdt.save_json(info, DIRS["results"] / f"{label}_optuna_best.json", indent=1)
    run = wb_init(f"{label}-optuna-summary", "optuna-summary", {"search_space": space})
    if run:
        run.summary.update({"best_value": st.best_value, "best_trial": st.best_trial.number, "states": states,
                            **{f"best/{k}": v for k, v in st.best_params.items()}})
        run.log({"optuna_trials": wandb.Table(dataframe=df.astype(str))}); wb_log_figs(run, figs)
        art = wandb.Artifact(f"task2-{label}-study", type="optuna-study"); art.add_file(str(db)); run.log_artifact(art); run.finish()
    print("trial states:", states)
    print(f"best #{st.best_trial.number}: {st.best_value:.4f}", json.dumps(st.best_params))
    return info

In [ ]:
clf_study, clf_db = run_study(f"{TAG}_classifier", clf_objective, S["clf_trials"], S["clf_timeout"], CLF_BASELINE,
                              optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2))
clf_info = summarise_study(clf_study, clf_db, CLF_SPACE, CLF_BASELINE, "classifier")

In [ ]:
CLF_HP = dict(clf_study.best_params)
CLF_CKPT = DIRS["checkpoints"] / "task2_classifier_best.pth"
run = wb_init("clf-final", "clf-final", {**CLF_HP, "epochs": S["clf_final_epochs"], "patience": S["clf_patience"]})
clf, best_f1, clf_hist = train_classifier(CLF_HP, S["clf_final_epochs"], run, None, CLF_CKPT, S["clf_patience"])
ck = torch.load(CLF_CKPT, map_location=device, weights_only=False)
clf.load_state_dict(ck["model"]); clf.eval()
print(f"best epoch {ck['epoch']} | val macro-F1 {ck['val_macro_f1']:.4f}")
h = pd.DataFrame(clf_hist); h.to_csv(DIRS["results"] / "classifier_training_history.csv", index=False)
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
ax[0].plot(h.epoch, h["train/loss"], label="train"); ax[0].plot(h.epoch, h["val/loss"], label="val"); ax[0].set_title("cross-entropy"); ax[0].legend()
ax[1].plot(h.epoch, h["train/acc"], label="train acc"); ax[1].plot(h.epoch, h["val/acc"], label="val acc"); ax[1].plot(h.epoch, h["val/macro_f1"], label="val macro-F1"); ax[1].legend(); ax[1].set_title("accuracy / F1")
for n in C.CLASS_NAMES:
    ax[2].plot(h.epoch, h[f"val/recall/{n}"], label=n)
ax[2].set_title("validation recall per class"); ax[2].legend(fontsize=8)
for a in ax:
    a.axvline(ck["epoch"], color="grey", ls=":"); a.set_xlabel("epoch")
fig.tight_layout(); fig.savefig(DIRS["figures"] / "classifier_training_curves.png", dpi=150); plt.show()
if run:
    wb_log_figs(run, {"figures/classifier_training_curves": DIRS["figures"] / "classifier_training_curves.png"})
    art = wandb.Artifact("task2-classifier-checkpoint", type="model", metadata=CLF_HP); art.add_file(str(CLF_CKPT)); run.log_artifact(art)

## 4. Classifier evaluation (validation + test)
The test set is class-imbalanced: each image appears once clean and three times per corruption. Macro-averaged metrics and the row-normalised confusion matrix are therefore the fair summaries.

In [ ]:
test_images, test_row, test_failed = data.load_test_images()
test_records = [r for r in data.test_manifest["records"] if r["image"] in test_row]
if S["test_images"] is not None:
    keep = set(data.test_names[:S["test_images"]]); test_records = [r for r in test_records if r["image"] in keep]
test_loader = pdt.make_eval_loader(test_images, test_row, test_records, batch_size=128, num_workers=NUM_WORKERS)

yv, pv, _ = C.predict(clf, val_loader, device, AMP)
yt, pt, lt = C.predict(clf, test_loader, device, AMP)
mv, mt = C.classification_metrics(yv, pv), C.classification_metrics(yt, pt)
for split, m in [("validation", mv), ("test", mt)]:
    tab = C.per_class_table(m); tab.to_csv(DIRS["results"] / f"classifier_{split}_metrics.csv")
    print(f"\n{split}: accuracy {m['acc']:.4f} | macro P {m['macro_precision']:.4f} R {m['macro_recall']:.4f} F1 {m['macro_f1']:.4f}")
    display(tab.round(4))

fig, ax = plt.subplots(1, 2, figsize=(10, 4.3))
C.plot_confusion(mv["cm_norm"], mv["cm"], f"Validation (acc {mv['acc']:.3f})", ax=ax[0])
im = C.plot_confusion(mt["cm_norm"], mt["cm"], f"Test (acc {mt['acc']:.3f})", ax=ax[1])
fig.colorbar(im, ax=ax, fraction=0.03); fig.suptitle("Classifier — row-normalised confusion matrices (counts in brackets)")
fig.savefig(DIRS["figures"] / "classifier_confusion.png", dpi=150, bbox_inches="tight"); plt.show()

# Accuracy by corruption and severity on the test set (which severities are hard to detect?)
tsev = pd.DataFrame({"type": [r["type"] for r in test_records], "severity": [r["severity_level"] for r in test_records],
                     "correct": pt.argmax(1) == yt, "pred": [C.CLASS_NAMES[i] for i in pt.argmax(1)]})
acc_sev = tsev.groupby(["type", "severity"])["correct"].mean().rename("accuracy")
pred_dist = pd.crosstab([tsev.type, tsev.severity], tsev.pred, normalize="index")
acc_sev_tab = pd.concat([acc_sev, pred_dist], axis=1)
acc_sev_tab.to_csv(DIRS["results"] / "classifier_test_accuracy_by_severity.csv")
print("Test accuracy and prediction distribution by condition x severity:"); display(acc_sev_tab.round(3))
fig, ax = plt.subplots(figsize=(8, 3.2))
labels = [f"{t}\n{s}" for t, s in acc_sev.index]
ax.bar(range(len(acc_sev)), acc_sev.values, color=["C0" if t == "clean" else "C1" if t == "salt_pepper" else "C2" if t == "gaussian_blur" else "C3" for t, _ in acc_sev.index])
ax.set_xticks(range(len(acc_sev))); ax.set_xticklabels(labels, fontsize=7); ax.set_ylim(0, 1.02); ax.set_ylabel("accuracy")
ax.set_title("Classifier test accuracy by condition and severity"); fig.tight_layout()
fig.savefig(DIRS["figures"] / "classifier_accuracy_by_severity.png", dpi=150); plt.show()

if run:
    run.summary.update({**{f"val/{k}": v for k, v in mv.items() if not isinstance(v, np.ndarray)},
                        **{f"test/{k}": v for k, v in mt.items() if not isinstance(v, np.ndarray)}})
    run.log({"test/confusion_matrix": wandb.plot.confusion_matrix(y_true=yt.tolist(), preds=pt.argmax(1).tolist(), class_names=list(C.CLASS_NAMES)),
             "val/confusion_matrix": wandb.plot.confusion_matrix(y_true=yv.tolist(), preds=pv.argmax(1).tolist(), class_names=list(C.CLASS_NAMES)),
             "test/per_class": wandb.Table(dataframe=C.per_class_table(mt).reset_index()),
             "test/accuracy_by_severity": wandb.Table(dataframe=acc_sev_tab.reset_index())})
    wb_log_figs(run, {"figures/classifier_confusion": DIRS["figures"] / "classifier_confusion.png",
                      "figures/classifier_accuracy_by_severity": DIRS["figures"] / "classifier_accuracy_by_severity.png"})
    run.finish()

## 5. Specialist autoencoders
**Shared Optuna search.** Each trial trains all three specialists with the same hyperparameters, each only on its own corruption. The objective is the mean validation score of the three, each scored on its own corruption type.
Specialists are trained in the order occlusion → blur → salt-and-pepper, and every epoch is reported to the median pruner. Weak configurations are usually stopped during the first (hardest) specialist.
The search space and the **8× code rule** match Task 1 v2: lr, bottleneck (latent channels), encoder channels, batch size, L1/SSIM weight α, dropout, and bottlenecked skips.
Then the three final specialists are trained **independently** (separate initialisation, optimiser and checkpoint) with the selected configuration.

In [ ]:
SPEC_SPACE = {
    "lr":              {"type": "float", "low": 1e-4, "high": 3e-3, "log": True},
    "batch_size":      {"type": "categorical", "choices": [16, 32, 64]},
    "latent_channels": {"type": "categorical", "choices": [8, 16, 32, 64]},
    "base_channels":   {"type": "categorical", "choices": [32, 48, 64]},
    "dropout":         {"type": "float", "low": 0.0, "high": 0.3},
    "alpha":           {"type": "float", "low": 0.5, "high": 0.95},
    "skip_levels":     {"type": "categorical", "choices": [0, 1, 2]},
    "skip_channels":   {"type": "categorical", "choices": [2, 4, 8]},
}
SPEC_BASELINE = {"lr": 1e-3, "batch_size": 32, "latent_channels": 32, "base_channels": 32, "dropout": 0.1, "alpha": 0.8,
                 "skip_levels": 0, "skip_channels": 4}

def build_ae(hp):
    return R.ConvAutoencoder(hp["base_channels"], hp["latent_channels"], hp["dropout"], hp["skip_levels"], hp["skip_channels"]).to(device)

def train_specialist(k, hp, epochs, run=None, trial=None, step0=0, ckpt=None, patience=None, verbose=False):
    torch.manual_seed(SEED + 100 * k + (trial.number if trial else 0))
    model = build_ae(hp)
    loader = pdt.make_train_loader(train_images, batch_size=hp["batch_size"], classes=(k,), num_workers=NUM_WORKERS)
    loss_fn = R.L1SSIMLoss(hp["alpha"])
    opt, sch = R.make_optimizer_and_scheduler(model, hp["lr"], SPEC_WEIGHT_DECAY, len(loader), epochs, warmup_epochs=1 if epochs < 10 else 2)
    scaler = R.make_scaler(device, AMP)
    best, best_ep, hist = -1.0, -1, []
    try:
        for ep in range(epochs):
            t0 = time.time()
            tr = R.train_one_epoch(model, loader, loss_fn, opt, sch, device, scaler, AMP)
            ev = R.evaluate(model, val_loader_by_type[k], device, loss_fn, AMP)
            rec = {"epoch": ep, "lr": sch.get_last_lr()[0], "time_s": time.time() - t0, **{f"train/{a}": b for a, b in tr.items()},
                   "val/loss": ev["loss"], "val/score": ev["score"], "val/psnr": ev["psnr"], "val/ssim": ev["ssim"], "val/mae": ev["mae"]}
            hist.append(rec)
            if run:
                run.log({f"{EXPERTS[k]}/{a}": b for a, b in rec.items()} if trial else rec)
            if ev["score"] > best:
                best, best_ep = ev["score"], ep
                if ckpt:
                    torch.save({"model": model.state_dict(), "model_config": model.config, "hp": hp, "expert": EXPERTS[k],
                                "label": k, "epoch": ep, "val": {a: b for a, b in ev.items() if a != "per_image"}}, ckpt)
            if verbose:
                print(f"[{EXPERTS[k]}] ep {ep:3d} | train {tr['loss']:.4f} | val {ev['loss']:.4f} PSNR {ev['psnr']:.2f} "
                      f"SSIM {ev['ssim']:.4f}{' *' if best_ep == ep else ''} | {rec['time_s']:.0f}s")
            if trial:
                trial.report(ev["score"], step0 + ep)
                if trial.should_prune():
                    raise optuna.TrialPruned()
            if patience and ep - best_ep >= patience:
                print(f"[{EXPERTS[k]}] early stopping"); break
    finally:
        del loader
    return model, best, hist

def spec_objective(trial):
    hp = suggest(trial, SPEC_SPACE)
    probe = R.ConvAutoencoder(8, hp["latent_channels"], 0, hp["skip_levels"], hp["skip_channels"])
    trial.set_user_attr("code_dim", probe.code_dim)
    if probe.code_dim > MAX_CODE:
        trial.set_user_attr("invalid", f"code {probe.code_dim} > {MAX_CODE}")
        raise optuna.TrialPruned("invalid code size")
    run = wb_init(f"spec-trial-{trial.number:03d}", "spec-trial", {**hp, "trial": trial.number, "code_dim": probe.code_dim})
    E, scores, state = S["spec_trial_epochs"], {}, "complete"
    try:
        for i, k in enumerate(SPEC_ORDER):
            m, best, _ = train_specialist(k, hp, E, run, trial, step0=i * E)
            scores[EXPERTS[k]] = best
            trial.set_user_attr(f"score_{EXPERTS[k]}", best)
            del m; gc.collect(); torch.cuda.empty_cache()
    except optuna.TrialPruned:
        state = "pruned"; raise
    except torch.cuda.OutOfMemoryError:
        state = "oom"; raise optuna.TrialPruned("OOM")
    finally:
        if run:
            run.summary.update({"state": state, **{f"score/{a}": b for a, b in scores.items()}}); run.finish()
        gc.collect(); torch.cuda.empty_cache()
    return float(np.mean(list(scores.values())))

In [ ]:
spec_study, spec_db = run_study(f"{TAG}_specialists", spec_objective, S["spec_trials"], S["spec_timeout"], SPEC_BASELINE,
                                optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=max(1, S["spec_trial_epochs"] - 2)))
spec_info = summarise_study(spec_study, spec_db, SPEC_SPACE, SPEC_BASELINE, "specialists")

In [ ]:
SPEC_HP = dict(spec_study.best_params)
specialists, spec_hist = {}, {}
for k in (1, 2, 3):
    ckpt = DIRS["checkpoints"] / f"task2_expert_{EXPERTS[k]}_best.pth"
    run = wb_init(f"spec-final-{EXPERTS[k]}", "spec-final", {**SPEC_HP, "expert": EXPERTS[k], "epochs": S["spec_final_epochs"]})
    t0 = time.time()
    m, best, hist = train_specialist(k, SPEC_HP, S["spec_final_epochs"], run, None, ckpt=ckpt, patience=S["spec_patience"], verbose=True)
    ck = torch.load(ckpt, map_location=device, weights_only=False)
    m.load_state_dict(ck["model"]); m.eval(); specialists[k] = m; spec_hist[k] = pd.DataFrame(hist)
    spec_hist[k].to_csv(DIRS["results"] / f"specialist_{EXPERTS[k]}_history.csv", index=False)
    print(f"[{EXPERTS[k]}] done in {(time.time() - t0) / 60:.1f} min | best epoch {ck['epoch']} | val PSNR {ck['val']['psnr']:.2f} SSIM {ck['val']['ssim']:.4f}")
    if run:
        run.summary.update({"best_epoch": ck["epoch"], **{f"best_val/{a}": b for a, b in ck["val"].items()}, "code_dim": m.code_dim})
        art = wandb.Artifact(f"task2-expert-{EXPERTS[k]}-checkpoint", type="model", metadata={**SPEC_HP, "expert": EXPERTS[k]})
        art.add_file(str(ckpt)); run.log_artifact(art); run.finish()
    gc.collect(); torch.cuda.empty_cache()

# independence check: parameters must differ between specialists
p = [torch.cat([q.flatten() for q in specialists[k].parameters()]) for k in (1, 2, 3)]
print("specialists have independent parameters:", not torch.equal(p[0], p[1]) and not torch.equal(p[1], p[2]))

fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
for k, c in zip((1, 2, 3), ("C1", "C2", "C3")):
    h = spec_hist[k]
    ax[0].plot(h.epoch, h["train/loss"], color=c, ls=":"); ax[0].plot(h.epoch, h["val/loss"], color=c, label=EXPERTS[k])
    ax[1].plot(h.epoch, h["val/psnr"], color=c, label=EXPERTS[k]); ax[2].plot(h.epoch, h["val/ssim"], color=c, label=EXPERTS[k])
ax[0].set_title("loss (dotted = train)"); ax[1].set_title("val PSNR on own corruption"); ax[2].set_title("val SSIM on own corruption")
for a in ax:
    a.legend(fontsize=8); a.set_xlabel("epoch")
fig.tight_layout(); fig.savefig(DIRS["figures"] / "specialists_training_curves.png", dpi=150); plt.show()

## 6. Hard-routed restoration on the test set
Every test record is passed through the classifier and **all** branches (identity + 3 experts), and the per-image metrics are stored:
* **Oracle routing:** the true label (from the test manifest) selects the branch, which measures the specialists' own ability.
* **Predicted routing:** $\arg\max_k p_k$ selects the branch, which measures the full operational system.
* **Cross-expert matrix:** every branch on every true condition, i.e. what misrouting costs.
* **Universal (Task 1):** the Task 1 v2 model on the same records, if it is attached.

A clean input routed to the identity branch is reproduced exactly (PSNR = ∞), so clean rows report SSIM, MAE and the identity rate instead of a PSNR mean.

In [ ]:
universal = None
if T1_CKPT is not None:
    ck1 = torch.load(T1_CKPT, map_location=device, weights_only=False)
    universal = R.ConvAutoencoder(**ck1["model_config"]).to(device); universal.load_state_dict(ck1["model"]); universal.eval()

@torch.no_grad()
def route_all(records, images, row):
    loader = pdt.make_eval_loader(images, row, records, batch_size=128, return_index=True, num_workers=NUM_WORKERS)
    N = len(records)
    out = {k: np.zeros((N, 4), np.float32) for k in ("psnr", "ssim", "mae")}
    out.update({"probs": np.zeros((N, 4), np.float32), "label": np.zeros(N, int)})
    if universal is not None:
        out.update({f"u_{k}": np.zeros(N, np.float32) for k in ("psnr", "ssim", "mae")})
    use_amp = AMP and device.type == "cuda"
    for x, y, lab, idx in loader:
        x, y = x.to(device), y.to(device); idx = idx.numpy()
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            probs = torch.softmax(clf(x).float(), 1)
            branches = [x] + [specialists[k](x) for k in (1, 2, 3)]
            if universal is not None:
                u = universal(x)
        out["probs"][idx] = probs.cpu().numpy(); out["label"][idx] = lab.numpy()
        for b, o in enumerate(branches):
            o = o.float().clamp(0, 1)
            out["psnr"][idx, b] = R.psnr_per_image(o, y).cpu().numpy()
            out["ssim"][idx, b] = R.ssim_per_image(o, y).cpu().numpy()
            out["mae"][idx, b] = R.mae_per_image(o, y).cpu().numpy()
        if universal is not None:
            u = u.float().clamp(0, 1)
            out["u_psnr"][idx] = R.psnr_per_image(u, y).cpu().numpy(); out["u_ssim"][idx] = R.ssim_per_image(u, y).cpu().numpy()
            out["u_mae"][idx] = R.mae_per_image(u, y).cpu().numpy()
    return out

t0 = time.time()
RT = route_all(test_records, test_images, test_row)
print(f"routed {len(test_records)} test records through classifier + 4 branches{' + universal' if universal is not None else ''} in {time.time() - t0:.0f}s")

lab, pred = RT["label"], RT["probs"].argmax(1)
ii = np.arange(len(lab))
df = pd.DataFrame({"id": [r["id"] for r in test_records], "image": [r["image"] for r in test_records],
                   "type": [r["type"] for r in test_records], "severity": [r["severity_level"] for r in test_records],
                   "label": lab, "pred": pred, "p_max": RT["probs"].max(1), "misrouted": lab != pred,
                   **{f"p_{n}": RT["probs"][:, i] for i, n in enumerate(C.CLASS_NAMES)}})
for m in ("psnr", "ssim", "mae"):
    df[f"in_{m}"] = RT[m][:, 0]
    df[f"oracle_{m}"] = RT[m][ii, lab]
    df[f"pred_{m}"] = RT[m][ii, pred]
    if universal is not None:
        df[f"univ_{m}"] = RT[f"u_{m}"]
# identity output on a clean input is exact -> PSNR infinite; mark as NaN so it never enters a mean
for c in ("in_psnr", "oracle_psnr", "pred_psnr"):
    df.loc[df[c] >= 99.9, c] = np.nan
df.to_csv(DIRS["results"] / "routing_test_per_record.csv", index=False)
print(f"misrouted records: {df.misrouted.sum()} / {len(df)} ({df.misrouted.mean():.1%})")

In [ ]:
order = ["clean", "salt_pepper", "gaussian_blur", "occlusion"]
sys_cols = ["in", "oracle", "pred"] + (["univ"] if universal is not None else [])
names = {"in": "input", "oracle": "oracle", "pred": "predicted", "univ": "universal (T1)"}

def summary(g):
    row = {"n": len(g), "routing acc": 1 - g.misrouted.mean()}
    for s in sys_cols:
        row[f"PSNR {names[s]}"] = g[f"{s}_psnr"].mean()
        row[f"SSIM {names[s]}"] = g[f"{s}_ssim"].mean()
    row["ΔPSNR pred-oracle"] = (g.pred_psnr - g.oracle_psnr).mean()
    return pd.Series(row)

by_type = df.groupby("type").apply(summary).reindex(order)
by_type.loc["all corrupted"] = summary(df[df.type != "clean"])
clean = df[df.type == "clean"]
by_type.loc["clean", "identity rate"] = (clean.pred == 0).mean()
by_sev = df.groupby(["type", "severity"]).apply(summary)
# Clean rows: identity output is exact (PSNR = ∞), so PSNR means over the remaining (misrouted) images would mislead
for tab_ in (by_type, by_sev):
    idx_ = [i for i in tab_.index if (i == "clean" or (isinstance(i, tuple) and i[0] == "clean"))]
    tab_.loc[idx_, ["PSNR input", "PSNR oracle", "PSNR predicted", "ΔPSNR pred-oracle"]] = np.nan
    if len(idx_):
        tab_.loc[idx_, "PSNR predicted (misrouted clean only)"] = df[(df.type == "clean") & df.misrouted].pred_psnr.mean()
by_sev = by_sev.reindex([(t, s) for t in order for s in ["none", "low", "medium", "high"] if (t, s) in by_sev.index])
print("Hard routing on the test set, by condition (clean PSNR is ∞ under identity — see SSIM / identity rate):")
display(by_type.round(3)); print("By condition x severity:"); display(by_sev.round(3))
by_type.to_csv(DIRS["results"] / "routing_test_by_type.csv"); by_sev.to_csv(DIRS["results"] / "routing_test_by_type_severity.csv")
for nm, t in [("routing_test_by_type", by_type), ("routing_test_by_type_severity", by_sev)]:
    try:
        (DIRS["results"] / f"{nm}.tex").write_text(t.round(3).to_latex(float_format="%.3f"))
    except Exception as e:
        print("LaTeX skipped:", e)

# Cross-expert matrix: mean SSIM / PSNR of every branch on every true condition
branch_names = ["identity", "A_salt", "A_blur", "A_occ"]
cross_ssim = pd.DataFrame([[RT["ssim"][lab == t, b].mean() for b in range(4)] for t in range(4)], index=order, columns=branch_names)
ps = RT["psnr"].copy(); ps[ps >= 99.9] = np.nan          # identity on clean = exact copy (∞ dB)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    cross_psnr = pd.DataFrame([[np.nanmean(ps[lab == t, b]) for b in range(4)] for t in range(4)], index=order, columns=branch_names)
cross_ssim.to_csv(DIRS["results"] / "cross_expert_ssim.csv"); cross_psnr.to_csv(DIRS["results"] / "cross_expert_psnr.csv")
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, (mat, t, fmt) in zip(ax, [(cross_psnr, "PSNR (dB)", "{:.1f}"), (cross_ssim, "SSIM", "{:.3f}")]):
    v = mat.values; im = a.imshow(np.nan_to_num(v, nan=np.nanmax(v)), cmap="viridis")
    a.set_xticks(range(4)); a.set_xticklabels(branch_names); a.set_yticks(range(4)); a.set_yticklabels(order)
    a.set_xlabel("branch used"); a.set_ylabel("true condition"); a.set_title(f"Cross-expert matrix — {t}")
    for i in range(4):
        for j in range(4):
            a.text(j, i, "∞" if np.isnan(v[i, j]) else fmt.format(v[i, j]), ha="center", va="center",
                   color="white" if (not np.isnan(v[i, j]) and v[i, j] < np.nanmean(v)) else "black", fontsize=9)
            if i == j:
                a.add_patch(plt.Rectangle((j - .5, i - .5), 1, 1, fill=False, ec="red", lw=2))
fig.tight_layout(); fig.savefig(DIRS["figures"] / "cross_expert_matrix.png", dpi=150); plt.show()

# Misrouting cost: for each (true, predicted) pair, how often and how much it costs vs oracle
mis = df[df.misrouted].copy()
mis["pair"] = [f"{C.CLASS_NAMES[a]} → {C.CLASS_NAMES[b]}" for a, b in zip(mis.label, mis.pred)]
mis["d_ssim"] = mis.pred_ssim - mis.oracle_ssim
mis["d_psnr"] = mis.pred_psnr - mis.oracle_psnr
cost = mis.groupby("pair").agg(count=("id", "size"), mean_d_ssim=("d_ssim", "mean"), mean_d_psnr=("d_psnr", "mean"),
                               worse_than_input=("pred_ssim", lambda s: float((s < mis.loc[s.index, "in_ssim"]).mean()))).sort_values("count", ascending=False)
cost.to_csv(DIRS["results"] / "misrouting_cost.csv")
print("Misrouting cost (predicted − oracle; negative = classifier error hurt the restoration):"); display(cost.round(3))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
sev = df[df.type != "clean"].groupby(["type", "severity"])
for i, (m, lab_) in enumerate([("psnr", "PSNR (dB)"), ("ssim", "SSIM")]):
    for t, c in zip(order[1:], ["C1", "C2", "C3"]):
        g = sev[[f"{s}_{m}" for s in sys_cols]].mean()
        ax[i].plot(["low", "medium", "high"], [g.loc[(t, s), f"in_{m}"] for s in ("low", "medium", "high")], ":x", color=c)
        ax[i].plot(["low", "medium", "high"], [g.loc[(t, s), f"oracle_{m}"] for s in ("low", "medium", "high")], "-o", color=c, label=f"{t} oracle")
        ax[i].plot(["low", "medium", "high"], [g.loc[(t, s), f"pred_{m}"] for s in ("low", "medium", "high")], "--s", color=c, alpha=.7, label=f"{t} predicted")
        if universal is not None:
            ax[i].plot(["low", "medium", "high"], [g.loc[(t, s), f"univ_{m}"] for s in ("low", "medium", "high")], "-.^", color=c, alpha=.5, label=f"{t} universal")
    ax[i].set_ylabel(lab_); ax[i].set_xlabel("severity")
ax[1].legend(fontsize=6, ncol=3); fig.suptitle("Hard routing vs severity (dotted = corrupted input)")
fig.tight_layout(); fig.savefig(DIRS["figures"] / "routing_severity_curves.png", dpi=150); plt.show()

## 7. Visual results: representative routed examples and classifier-induced failures

In [ ]:
rec_by_id = {r["id"]: r for r in test_records}
dfi = df.set_index("id")

@torch.no_grad()
def branch_outputs(recs):
    cor, cl = records_to_arrays(recs, test_images, test_row)
    x = torch.from_numpy(cor.transpose(0, 3, 1, 2)).to(device)
    outs = [cor] + [specialists[k](x).float().clamp(0, 1).cpu().numpy().transpose(0, 2, 3, 1) for k in (1, 2, 3)]
    return cor, cl, outs

def routing_grid(ids, path, title):
    recs = [rec_by_id[i] for i in ids]
    cor, cl, outs = branch_outputs(recs)
    fig, axes = plt.subplots(len(ids), 5, figsize=(12, 2.5 * len(ids)), squeeze=False)
    for r, i in enumerate(ids):
        d = dfi.loc[i]; lb, pr = int(d.label), int(d.pred)
        probs = " ".join(f"{n[:4]} {d[f'p_{n}']:.2f}" for n in C.CLASS_NAMES)
        panels = [(cl[r], "clean target"), (cor[r], f"input: {pdt.describe_params(recs[r])}\n{probs}"),
                  (outs[lb][r], f"oracle → {branch_names[lb]}\nSSIM {d.oracle_ssim:.3f}"),
                  (outs[pr][r], f"predicted → {branch_names[pr]}\nSSIM {d.pred_ssim:.3f}")]
        for c, (im, t) in enumerate(panels):
            axes[r, c].imshow(np.clip(im, 0, 1)); axes[r, c].set_title(t, fontsize=7)
        axes[r, 4].imshow(np.abs(outs[pr][r] - cl[r]).mean(-1), cmap="magma", vmin=0, vmax=.5); axes[r, 4].set_title("|error| predicted", fontsize=7)
        for a in axes[r]:
            a.set_xticks([]); a.set_yticks([])
        if lb != pr:
            for a in axes[r]:
                for s in a.spines.values():
                    s.set_edgecolor("red"); s.set_linewidth(2)
    fig.suptitle(title, fontsize=11); fig.tight_layout(); fig.savefig(path, dpi=140, bbox_inches="tight"); plt.show()
    return path

figs = {}
# Representative: per condition x severity, the correctly-routed record closest to the median predicted SSIM
rep = []
for t in order:
    for s in (["none"] if t == "clean" else ["low", "medium", "high"]):
        g = df[(df.type == t) & (df.severity == s) & ~df.misrouted]
        if len(g):
            rep.append(g.iloc[(g.pred_ssim - g.pred_ssim.median()).abs().argsort().iloc[0]]["id"])
figs["figures/routing_representative"] = routing_grid(rep[:10], DIRS["figures"] / "routing_representative.png",
                                                      "Hard routing — representative correctly-routed test examples")
# Classifier-induced failures: misrouted records with the largest SSIM loss vs oracle, one per (true, pred) pair
if len(mis):
    worst = mis.sort_values("d_ssim").drop_duplicates("pair").head(6)
    worst[["type", "severity", "pair", "p_max", "oracle_ssim", "pred_ssim", "d_ssim", "d_psnr"]].to_csv(DIRS["results"] / "misrouting_failures.csv")
    display(worst[["type", "severity", "pair", "p_max", "oracle_ssim", "pred_ssim", "d_ssim"]].round(3))
    figs["figures/misrouting_failures"] = routing_grid(list(worst.id), DIRS["figures"] / "misrouting_failures.png",
                                                       "Classifier errors that caused restoration failures (red = misrouted)")
    # Harmless misroutings (e.g. low blur sent to identity): predicted >= oracle
    ok_mis = mis[mis.d_ssim >= 0]
    print(f"misroutings that did NOT hurt (predicted SSIM ≥ oracle): {len(ok_mis)} / {len(mis)} "
          f"— most common pairs: {ok_mis.pair.value_counts().head(3).to_dict()}")

run = wb_init("routing-eval", "eval", {"clf_hp": CLF_HP, "spec_hp": SPEC_HP, "universal_attached": universal is not None})
if run:
    run.summary.update({"routing/misrouted_frac": float(df.misrouted.mean()),
                        **{f"routing/{c}": float(v) for c, v in by_type.loc["all corrupted"].items() if pd.notna(v)}})
    run.log({"routing/by_type": wandb.Table(dataframe=by_type.reset_index().rename(columns={"index": "type"})),
             "routing/by_type_severity": wandb.Table(dataframe=by_sev.reset_index()),
             "routing/cross_expert_psnr": wandb.Table(dataframe=cross_psnr.reset_index()),
             "routing/cross_expert_ssim": wandb.Table(dataframe=cross_ssim.reset_index()),
             "routing/misrouting_cost": wandb.Table(dataframe=cost.reset_index())})
    wb_log_figs(run, {**figs, "figures/cross_expert_matrix": DIRS["figures"] / "cross_expert_matrix.png",
                      "figures/routing_severity_curves": DIRS["figures"] / "routing_severity_curves.png",
                      "figures/specialists_training_curves": DIRS["figures"] / "specialists_training_curves.png"})

## 8. ONNX export: classifier + three specialists

In [ ]:
onnx_files, parity, latency = {}, {}, {}
vx = torch.cat([b[0] for _, b in zip(range(2), val_loader)])[:64]
tx = torch.cat([b[0] for _, b in zip(range(1), test_loader)])[:64]

clf_cpu = C.CorruptionClassifier(**{k: v for k, v in clf.config.items()}).cpu()
clf_cpu.load_state_dict({k: v.cpu() for k, v in clf.state_dict().items()}); clf_cpu.eval()
p = DIRS["models"] / "task2_classifier.onnx"
export_onnx_multi(C.ClassifierWithProbs(clf_cpu), p, ["logits", "probs"])
sess = ort.InferenceSession(str(p), providers=["CPUExecutionProvider"])
with torch.no_grad():
    ref_l = clf_cpu(tx)
o_l, o_p = sess.run(None, {"input": tx.numpy()})
parity["classifier"] = {"max_abs_diff_logits": float(np.abs(o_l - ref_l.numpy()).max()),
                        "argmax_agreement": float((o_p.argmax(1) == ref_l.argmax(1).numpy()).mean())}
parity["classifier"]["passed"] = parity["classifier"]["max_abs_diff_logits"] < 1e-3 and parity["classifier"]["argmax_agreement"] == 1.0
onnx_files["classifier"] = p.name
latency["classifier"] = R.onnx_latency_ms(p, n_runs=100 if MODE == "full" else 20)

for k in (1, 2, 3):
    m_cpu = R.ConvAutoencoder(**specialists[k].config).cpu()
    m_cpu.load_state_dict({a: b.cpu() for a, b in specialists[k].state_dict().items()}); m_cpu.eval()
    p = DIRS["models"] / f"task2_expert_{EXPERTS[k]}.onnx"
    R.export_onnx(m_cpu, p)
    parity[EXPERTS[k]] = {"val": R.verify_onnx(m_cpu, p, vx), "test": R.verify_onnx(m_cpu, p, tx)}
    parity[EXPERTS[k]]["passed"] = parity[EXPERTS[k]]["val"]["passed"] and parity[EXPERTS[k]]["test"]["passed"]
    onnx_files[EXPERTS[k]] = p.name
    latency[EXPERTS[k]] = R.onnx_latency_ms(p, n_runs=100 if MODE == "full" else 20)
print(json.dumps({"parity": parity, "latency_ms": {k: round(v["median_ms"], 1) for k, v in latency.items()}}, indent=1))
assert all(v["passed"] for v in parity.values()), "ONNX parity failed"

sidecar = {"task": "hard_routed_restoration", "class_names": list(C.CLASS_NAMES), "onnx_files": onnx_files,
           "routing": {"clean": "identity", "salt_pepper": onnx_files["salt_pepper"], "gaussian_blur": onnx_files["gaussian_blur"],
                       "occlusion": onnx_files["occlusion"]},
           "classifier_outputs": {"logits": "[batch, 4]", "probs": "[batch, 4] softmax"},
           "input": {"name": "input", "shape": ["batch", 3, 128, 128], "range": [0, 1], "layout": "NCHW RGB",
                     "preprocess": "PIL convert('RGB') -> resize 128x128 BICUBIC -> /255"},
           "classifier_hp": CLF_HP, "classifier_config": clf.config, "specialist_hp": SPEC_HP,
           "specialist_config": specialists[1].config, "onnx_parity": parity, "cpu_latency_ms": latency,
           "test_classifier": {k: mt[k] for k in ("acc", "macro_precision", "macro_recall", "macro_f1")}}
pdt.save_json(sidecar, DIRS["models"] / "task2_hard_routing.json", indent=1)
if run:
    run.summary.update({f"onnx/{k}_passed": v["passed"] for k, v in parity.items()})
    art = wandb.Artifact("task2-hard-routing-onnx", type="model", metadata=sidecar)
    for f in list(onnx_files.values()) + ["task2_hard_routing.json"]:
        art.add_file(str(DIRS["models"] / f))
    run.log_artifact(art); run.finish()

## 9. Summary

In [ ]:
summary = {"mode": MODE, "data_source": data.source,
           "classifier": {"best_trial": clf_info["best_trial"], "hp": CLF_HP, "val": {k: mv[k] for k in ("acc", "macro_f1")},
                          "test": {k: mt[k] for k in ("acc", "macro_precision", "macro_recall", "macro_f1")}},
           "specialists": {"best_trial": spec_info["best_trial"], "hp": SPEC_HP, "code_dim": specialists[1].code_dim},
           "routing_all_corrupted": by_type.loc["all corrupted"].dropna().round(4).to_dict(),
           "clean_identity_rate": float(by_type.loc["clean", "identity rate"]),
           "misrouted_frac": float(df.misrouted.mean()), "onnx_parity_passed": all(v["passed"] for v in parity.values()),
           "unit_tests_passed": int(tests_df.passed.sum())}
pdt.save_json(summary, DIRS["results"] / "summary.json", indent=1)
print(json.dumps(summary, indent=1, default=str))
print("\nOutputs:")
for d in DIRS.values():
    for p in sorted(d.rglob("*")):
        if p.is_file():
            print(f"  {p.relative_to(WORK_DIR)}  ({p.stat().st_size / 1e6:.2f} MB)")
print("\nSMOKE TEST PASSED — switch MODE to 'full' and Save & Run All." if MODE == "smoke" else "\nTASK 2 COMPLETE.")